# Soccer state-space trading backtest

This notebook builds and validates a state-space soccer trading framework. It starts from cached API-Football data, cleans event/fixture tables, builds pre-match and in-play state features, fits scoring-hazard models, constructs synthetic live market probabilities, and evaluates the resulting betting strategy against dumb baselines and stress tests.

# Data acquisition and cache setup
This cell defines the API-Football pull, local cache, and normalized raw data tables. It is kept as the reproducible acquisition layer; later cells use the saved parquet artifacts.


In [ ]:
import json
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import deque
from pathlib import Path
from datetime import datetime
from getpass import getpass
import pandas as pd
import requests

API_KEY = getpass("API-Football key: ").strip()
BASE = "https://v3.football.api-sports.io"
HEADERS = {"x-apisports-key": API_KEY}

PROJECT = Path("/content/drive/MyDrive/soccer-betting")
RAW = PROJECT / "data" / "raw" / "api_football"
PROCESSED = PROJECT / "data" / "processed" / "api_football"
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
(RAW / "events").mkdir(exist_ok=True)
(RAW / "statistics").mkdir(exist_ok=True)
(RAW / "fixtures").mkdir(exist_ok=True)

LEAGUES = {
    "premier_league":     39,
    "laliga":             140,
    "serie_a":            135,
    "bundesliga":         78,
    "ligue_1":            61,
    "turkish_super_lig":  203,
    "champions_league":   2,
    "europa_league":      3,
}
SEASONS = [2023, 2024, 2025]

def log(msg, indent=0):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {'  '*indent}{msg}", flush=True)

class RateLimiter:
    def __init__(self, max_requests, window_seconds):
        self.max_requests = max_requests
        self.window = window_seconds
        self.timestamps = deque()
        self.lock = threading.Lock()
        self.penalty_until = 0.0

    def acquire(self):
        while True:
            with self.lock:
                now = time.time()
                if now < self.penalty_until:
                    wait = self.penalty_until - now
                else:
                    while self.timestamps and self.timestamps[0] < now - self.window:
                        self.timestamps.popleft()
                    if len(self.timestamps) < self.max_requests:
                        self.timestamps.append(now)
                        return
                    wait = self.timestamps[0] + self.window - now
            if wait > 0:
                time.sleep(min(wait, 0.1))

    def penalize(self, seconds):
        with self.lock:
            self.penalty_until = max(self.penalty_until, time.time() + seconds)

limiter = RateLimiter(max_requests=600, window_seconds=60)

def api_get(endpoint, params=None, cache_key=None, max_retries=5):
    if cache_key:
        cache_path = RAW / f"{cache_key}.json"
        if cache_path.exists():
            return json.loads(cache_path.read_text())

    for attempt in range(max_retries):
        limiter.acquire()
        try:
            r = requests.get(f"{BASE}{endpoint}", headers=HEADERS, params=params, timeout=30)
        except Exception:
            time.sleep(1.0)
            continue

        if r.status_code == 429:
            backoff = min(5 * (2 ** attempt), 60)
            limiter.penalize(backoff)
            time.sleep(backoff)
            continue

        if r.status_code != 200:
            time.sleep(1.0)
            continue

        try:
            data = r.json()
        except Exception:
            time.sleep(0.5)
            continue

        if data.get("errors"):
            errs = str(data["errors"])
            if "rateLimit" in errs or "Too many" in errs:
                backoff = min(5 * (2 ** attempt), 60)
                limiter.penalize(backoff)
                time.sleep(backoff)
                continue
            if cache_key:
                cache_path.parent.mkdir(parents=True, exist_ok=True)
                cache_path.write_text(json.dumps(data))
            return data

        if cache_key:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            cache_path.write_text(json.dumps(data))
        return data

    return {"response": [], "errors": ["max_retries"]}

def get_fixtures(league_id, season, slug):
    return api_get("/fixtures", params={"league": league_id, "season": season},
                   cache_key=f"fixtures/{slug}_{season}").get("response", [])

def get_events(fid):
    return api_get("/fixtures/events", params={"fixture": fid},
                   cache_key=f"events/event_{fid}").get("response", [])

def get_statistics(fid):
    return api_get("/fixtures/statistics", params={"fixture": fid},
                   cache_key=f"statistics/stats_{fid}").get("response", [])

def normalize_fixture(fx, slug, season):
    f = fx["fixture"]
    return {
        "fixture_id": f["id"], "target_slug": slug, "season": season,
        "date": f["date"], "timestamp": f["timestamp"],
        "venue_name": (f.get("venue") or {}).get("name"),
        "venue_city": (f.get("venue") or {}).get("city"),
        "status": f["status"]["short"], "elapsed": f["status"].get("elapsed"),
        "round": fx["league"].get("round"),
        "home_team_id": fx["teams"]["home"]["id"],
        "home_team_name": fx["teams"]["home"]["name"],
        "away_team_id": fx["teams"]["away"]["id"],
        "away_team_name": fx["teams"]["away"]["name"],
        "home_score": fx["goals"]["home"], "away_score": fx["goals"]["away"],
        "ht_home_score": fx["score"]["halftime"]["home"],
        "ht_away_score": fx["score"]["halftime"]["away"],
    }

def normalize_event(ev, fid):
    return {
        "fixture_id": fid, "minute": ev["time"]["elapsed"], "extra": ev["time"].get("extra"),
        "team_id": ev["team"]["id"], "team_name": ev["team"]["name"],
        "player_id": ev["player"].get("id"), "player_name": ev["player"].get("name"),
        "assist_id": (ev.get("assist") or {}).get("id"),
        "assist_name": (ev.get("assist") or {}).get("name"),
        "type": ev["type"], "detail": ev["detail"], "comments": ev.get("comments"),
    }

def normalize_statistics(stats_resp, fid):
    rows = []
    for ts in stats_resp:
        row = {"fixture_id": fid, "team_id": ts["team"]["id"], "team_name": ts["team"]["name"]}
        for s in ts.get("statistics", []):
            key = s["type"].lower().replace(" ", "_").replace("%", "pct")
            row[key] = s["value"]
        rows.append(row)
    return rows

log("Collecting fixture lists for all leagues and seasons")

all_fixtures = []
for slug, lid in LEAGUES.items():
    for season in SEASONS:
        fxs = get_fixtures(lid, season, slug)
        for fx in fxs:
            all_fixtures.append((fx, slug, season))
        log(f"{slug} {season}: {len(fxs)}", indent=1)

fix_rows = [normalize_fixture(fx, slug, season) for fx, slug, season in all_fixtures]
pd.DataFrame(fix_rows).to_parquet(PROCESSED / "fixtures.parquet", index=False)
log(f"\nfixtures.parquet: {len(fix_rows)} rows")

finished = [(fx, slug, season) for fx, slug, season in all_fixtures
            if fx["fixture"]["status"]["short"] in ("FT", "AET", "PEN")]

remaining = []
for fx, slug, season in finished:
    fid = fx["fixture"]["id"]
    if not (RAW / f"events/event_{fid}.json").exists() or \
       not (RAW / f"statistics/stats_{fid}.json").exists():
        remaining.append((fx, slug, season))

log(f"\nFinished:  {len(finished)}")
log(f"Cached:    {len(finished) - len(remaining)}", indent=1)
log(f"Remaining: {len(remaining)}", indent=1)
log(f"Est time at 600/min: {len(remaining) * 2 / 600:.1f} min", indent=1)

def pull_one(item):
    fx, _, _ = item
    fid = fx["fixture"]["id"]
    try:
        get_events(fid); get_statistics(fid)
        return fid, None
    except Exception as e:
        return fid, str(e)

log("Pulling fixture-level events and statistics")

errors = []
n_done = 0
t0 = time.time()
last_t, last_n = t0, 0

if remaining:
    with ThreadPoolExecutor(max_workers=8) as ex:
        futures = {ex.submit(pull_one, item): item for item in remaining}
        for fut in as_completed(futures):
            fid, err = fut.result()
            n_done += 1
            if err:
                errors.append((fid, err))
            now = time.time()
            if (n_done % 100 == 0) or (now - last_t > 15):
                rate = n_done / (now - t0) * 60
                recent = (n_done - last_n) / max(now - last_t, 0.1) * 60
                eta = (len(remaining) - n_done) / max(rate / 60, 0.1) / 60
                pct = n_done / len(remaining) * 100
                bar = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
                log(f"[{bar}] {n_done:>5}/{len(remaining)} ({pct:5.1f}%) "
                    f"avg={rate:>4.0f}/min recent={recent:>4.0f}/min "
                    f"eta={eta:>4.1f}min err={len(errors)}", indent=1)
                last_t, last_n = now, n_done

log(f"\nDone in {(time.time()-t0)/60:.1f} min, errors={len(errors)}")

log("Building events.parquet and statistics.parquet")

all_events, all_stats = [], []
for ev_file in (RAW / "events").glob("event_*.json"):
    fid = int(ev_file.stem.split("_")[1])
    try:
        data = json.loads(ev_file.read_text())
        for ev in data.get("response", []):
            all_events.append(normalize_event(ev, fid))
    except Exception as e:
        log(f"bad events file {ev_file.name}: {e}", indent=1)

for st_file in (RAW / "statistics").glob("stats_*.json"):
    fid = int(st_file.stem.split("_")[1])
    try:
        data = json.loads(st_file.read_text())
        all_stats.extend(normalize_statistics(data.get("response", []), fid))
    except Exception as e:
        log(f"bad stats file {st_file.name}: {e}", indent=1)

events_df = pd.DataFrame(all_events)
stats_df = pd.DataFrame(all_stats)

def clean_pct(v):
    if isinstance(v, str) and v.endswith("%"):
        try: return float(v[:-1]) / 100
        except: return None
    return v

if not stats_df.empty:
    for col in ["ball_possession", "passes_pct"]:
        if col in stats_df.columns:
            stats_df[col] = stats_df[col].apply(clean_pct)
    if "expected_goals" in stats_df.columns:
        stats_df["expected_goals"] = pd.to_numeric(stats_df["expected_goals"], errors="coerce")

events_df.to_parquet(PROCESSED / "events.parquet", index=False)
stats_df.to_parquet(PROCESSED / "statistics.parquet", index=False)

log(f"\nevents.parquet:     {events_df.shape}", indent=1)
log(f"statistics.parquet: {stats_df.shape}", indent=1)
log(f"errors:             {len(errors)}", indent=1)
log(f"\nDONE")

# Data cleaning and fixture validation


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED = Path('/content/drive/MyDrive/soccer-betting/data/processed/api_football')

fixtures = pd.read_parquet(PROCESSED / 'fixtures.parquet')
events   = pd.read_parquet(PROCESSED / 'events.parquet')
stats    = pd.read_parquet(PROCESSED / 'statistics.parquet')

print(f"fixtures: {fixtures.shape}")
print(f"events:   {events.shape}")
print(f"stats:    {stats.shape}")

GALA_ID = 645   # API-Football's team_id for Galatasaray

gala_matches = fixtures[
    (fixtures['home_team_id'] == GALA_ID) | (fixtures['away_team_id'] == GALA_ID)
].copy()
gala_matches['date'] = pd.to_datetime(gala_matches['date'])
gala_matches = gala_matches.sort_values('date').reset_index(drop=True)

gala_matches['gala_home']     = gala_matches['home_team_id'] == GALA_ID
gala_matches['opponent']      = np.where(gala_matches['gala_home'],
                                          gala_matches['away_team_name'],
                                          gala_matches['home_team_name'])
gala_matches['gala_score']    = np.where(gala_matches['gala_home'],
                                          gala_matches['home_score'],
                                          gala_matches['away_score'])
gala_matches['opp_score']     = np.where(gala_matches['gala_home'],
                                          gala_matches['away_score'],
                                          gala_matches['home_score'])
gala_matches['gala_result']   = np.where(gala_matches['gala_score'] > gala_matches['opp_score'], 'W',
                                np.where(gala_matches['gala_score'] < gala_matches['opp_score'], 'L', 'D'))
gala_matches['venue']         = np.where(gala_matches['gala_home'], 'H', 'A')

print(f"\nGalatasaray matches: {len(gala_matches)}")
print(f"By competition (target_slug):")
print(gala_matches['target_slug'].value_counts())
print(f"\nBy season:")
print(gala_matches.groupby(['target_slug', 'season']).size())

print(f"\nResult distribution:")
print(gala_matches['gala_result'].value_counts())

print(f"\n=== Last 10 Galatasaray matches ===")
display(gala_matches.tail(10)[
    ['date', 'target_slug', 'venue', 'opponent', 'gala_score', 'opp_score', 'gala_result']
])

liv_match = gala_matches[
    gala_matches['opponent'].str.contains('Liverpool', case=False, na=False)
]

if len(liv_match) > 0:
    target_fid = int(liv_match.iloc[-1]['fixture_id'])
    print(f"\n=== Inspecting fixture {target_fid}: Galatasaray vs Liverpool ===")
else:
    ucl = gala_matches[gala_matches['target_slug'] == 'champions_league']
    if len(ucl) > 0:
        target_fid = int(ucl.iloc[-1]['fixture_id'])
        print(f"\n=== Inspecting most recent UCL fixture {target_fid} ===")
    else:
        target_fid = int(gala_matches.iloc[-1]['fixture_id'])
        print(f"\n=== Inspecting most recent fixture {target_fid} ===")

match_info = fixtures[fixtures['fixture_id'] == target_fid].iloc[0]
print(f"\n{match_info['date']}: {match_info['home_team_name']} {match_info['home_score']}-{match_info['away_score']} {match_info['away_team_name']}")
print(f"Competition: {match_info['target_slug']}, Season: {match_info['season']}")
print(f"Halftime: {match_info['ht_home_score']}-{match_info['ht_away_score']}")

match_events = events[events['fixture_id'] == target_fid].sort_values(['minute', 'extra']).copy()
print(f"\n=== Event timeline ({len(match_events)} events) ===")
display(match_events[['minute', 'extra', 'team_name', 'type', 'detail', 'player_name', 'assist_name']])

match_stats = stats[stats['fixture_id'] == target_fid]
print(f"\n=== Match statistics ===")
display(match_stats.T)

# Data integrity checks and cleaned tables

This section verifies the cached API pull and rebuilds the cleaned fixture, event, and feature tables used by the modeling pipeline.


In [ ]:
import pandas as pd
import numpy as np
import os
import shutil
import time
from pathlib import Path

PROJECT  = Path('/content/drive/MyDrive/soccer-betting')
PROCESSED = PROJECT / 'data' / 'processed' / 'api_football'

print("=== Verify Drive parquets ===")
for f in sorted(PROCESSED.glob('*.parquet')):
    print(f"  {f.name:<35s}  {f.stat().st_size/1e6:>7.2f} MB")

events = pd.read_parquet(PROCESSED / 'events.parquet')
stats  = pd.read_parquet(PROCESSED / 'statistics.parquet')
fixtures = pd.read_parquet(PROCESSED / 'fixtures.parquet')

print(f"\nfixtures: {fixtures.shape}, events: {events.shape}, stats: {stats.shape}")
assert events.shape[0] > 100_000, "events.parquet too small — pull failed"
assert stats.shape[0] > 10_000, "statistics.parquet too small — pull failed"

os.makedirs('/content/backup_parquets', exist_ok=True)
for fname in ['events.parquet', 'statistics.parquet', 'fixtures.parquet']:
    shutil.copy(PROCESSED / fname, f'/content/backup_parquets/{fname}')
print("\nBackups saved to /content/backup_parquets/")

fixtures['date'] = pd.to_datetime(fixtures['date'], utc=True)

shootout_fids = events[events['minute'] == 120].groupby('fixture_id').size().pipe(
    lambda s: s[s > 4].index.tolist()
)
events_clean = events[
    ~(events['fixture_id'].isin(shootout_fids) & (events['minute'] == 120))
    & ~events['detail'].fillna('').str.contains('cancelled|Missed', case=False, regex=True)
].copy()
events_clean.to_parquet(PROCESSED / 'events_clean.parquet', index=False)

closed = fixtures[fixtures['status'].isin(['FT', 'AET', 'PEN'])]
complete_stats_fids = stats.groupby('fixture_id').filter(lambda g: len(g) == 2)['fixture_id'].unique()
has_events_fids = events_clean['fixture_id'].unique()
usable = closed[closed['fixture_id'].isin(complete_stats_fids) & closed['fixture_id'].isin(has_events_fids)].copy()
usable.to_parquet(PROCESSED / 'fixtures_usable.parquet', index=False)

print(f"\nevents_clean: {len(events)} -> {len(events_clean)}")
print(f"fixtures_usable: {len(usable)}")

for fname in ['events_clean.parquet', 'fixtures_usable.parquet']:
    fpath = PROCESSED / fname
    with open(fpath, 'rb+') as fh:
        os.fsync(fh.fileno())
time.sleep(5)

print("\nCleaned tables are ready for feature reconstruction.")

# Pre-match feature construction and baseline model


# Pre-match feature reconstruction

This section rebuilds Dixon-Coles, Elo, standings, and gradient-boosted pre-match features.


In [ ]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from scipy.optimize import minimize
from scipy.stats import poisson
from math import lgamma
import xgboost as xgb
from sklearn.metrics import log_loss

PROCESSED = Path('/content/drive/MyDrive/soccer-betting/data/processed/api_football')

fixtures = pd.read_parquet(PROCESSED / 'fixtures_usable.parquet')
events_clean = pd.read_parquet(PROCESSED / 'events_clean.parquet')
stats = pd.read_parquet(PROCESSED / 'statistics.parquet')

fixtures['date'] = pd.to_datetime(fixtures['date'], utc=True)
fixtures = fixtures.sort_values('date').reset_index(drop=True)
fixtures['home_score'] = pd.to_numeric(fixtures['home_score'], errors='coerce')
fixtures['away_score'] = pd.to_numeric(fixtures['away_score'], errors='coerce')

print(f"loaded: fixtures {fixtures.shape}, events {events_clean.shape}, stats {stats.shape}")

def build_records(fixtures, stats):
    sc = ['fixture_id', 'team_id', 'shots_on_goal', 'total_shots', 'corner_kicks',
          'fouls', 'yellow_cards', 'red_cards', 'ball_possession', 'passes_pct',
          'goalkeeper_saves', 'expected_goals', 'shots_insidebox']
    sc = [c for c in sc if c in stats.columns]
    s = stats[sc].copy()
    for c in ['yellow_cards','red_cards','expected_goals','ball_possession','passes_pct']:
        if c in s.columns: s[c] = pd.to_numeric(s[c], errors='coerce')

    base = ['fixture_id','date','home_team_id','away_team_id',
            'home_team_name','away_team_name','home_score','away_score',
            'ht_home_score','ht_away_score','target_slug','season']
    h = fixtures[base].copy().rename(columns={
        'home_team_id':'team_id','away_team_id':'opp_id',
        'home_team_name':'team_name','away_team_name':'opp_name',
        'home_score':'goals_for','away_score':'goals_against',
        'ht_home_score':'ht_for','ht_away_score':'ht_against'})
    h['venue']='home'
    a = fixtures[base].copy().rename(columns={
        'away_team_id':'team_id','home_team_id':'opp_id',
        'away_team_name':'team_name','home_team_name':'opp_name',
        'away_score':'goals_for','home_score':'goals_against',
        'ht_away_score':'ht_for','ht_home_score':'ht_against'})
    a['venue']='away'
    r = pd.concat([h,a], ignore_index=True)
    for c in ['goals_for','goals_against','ht_for','ht_against']:
        r[c] = pd.to_numeric(r[c], errors='coerce')
    r['win']=(r['goals_for']>r['goals_against']).astype(float)
    r['draw']=(r['goals_for']==r['goals_against']).astype(float)
    r['loss']=(r['goals_for']<r['goals_against']).astype(float)
    r['points']=r['win']*3+r['draw']
    r['clean_sheet']=(r['goals_against']==0).astype(float)
    r['scored']=(r['goals_for']>0).astype(float)
    r['goal_diff']=r['goals_for']-r['goals_against']
    r['sh_for']=r['goals_for']-r['ht_for'].fillna(0)
    r['sh_against']=r['goals_against']-r['ht_against'].fillna(0)
    r = r.merge(s, on=['fixture_id','team_id'], how='left')
    return r.sort_values(['team_id','date']).reset_index(drop=True)

def build_event_feats(events):
    ev = events.copy()
    ev['minute'] = pd.to_numeric(ev['minute'], errors='coerce')
    rows = []
    for fid, grp in ev.groupby('fixture_id'):
        all_g = grp[(grp['type']=='Goal') &
                    ~grp['detail'].fillna('').str.contains('Own Goal|cancelled|Missed', case=False, regex=True)
                   ].sort_values(['minute','extra'])
        first = all_g.iloc[0]['team_name'] if len(all_g)>0 else None
        for team in grp['team_name'].dropna().unique():
            t = grp[grp['team_name']==team]
            tg = t[(t['type']=='Goal') &
                   ~t['detail'].fillna('').str.contains('Own Goal|cancelled|Missed', case=False, regex=True)]
            yel = (t['type']=='Card') & t['detail'].fillna('').str.contains('Yellow', na=False)
            red = (t['type']=='Card') & t['detail'].fillna('').str.contains('Red', na=False)
            rows.append({
                'fixture_id': fid, 'team_name': team,
                'first_goal_minute': tg['minute'].min() if len(tg) else np.nan,
                'last_goal_minute':  tg['minute'].max() if len(tg) else np.nan,
                'n_yellow': int(yel.sum()), 'n_red': int(red.sum()),
                'n_subs': int((t['type']=='subst').sum()),
                'scored_first': float(first==team) if first else 0.0,
                'late_goal': float((tg['minute']>=75).any()) if len(tg) else 0.0,
                'early_goal': float((tg['minute']<=15).any()) if len(tg) else 0.0,
            })
    return pd.DataFrame(rows)

records = build_records(fixtures, stats)
ev_feat = build_event_feats(events_clean)
records = records.merge(ev_feat, on=['fixture_id','team_name'], how='left')
for c in ['n_yellow','n_red','n_subs','scored_first','late_goal','early_goal']:
    records[c] = records[c].fillna(0)

ROLL = [c for c in ['goals_for','goals_against','goal_diff','points','win','draw','loss',
                    'clean_sheet','scored','sh_for','sh_against',
                    'shots_on_goal','total_shots','expected_goals','ball_possession',
                    'passes_pct','corner_kicks','fouls','yellow_cards',
                    'n_yellow','n_red','scored_first','late_goal','early_goal']
        if c in records.columns]

def roll(df, cols, w, mp):
    sh = df[cols].shift(1)
    out = sh.rolling(w, min_periods=mp).mean()
    out.columns = [f'L{w}_{c}' for c in cols]
    return out

def ewm(df, cols, alpha=0.3):
    sh = df[cols].shift(1)
    out = sh.ewm(alpha=alpha, min_periods=3, adjust=False).mean()
    out.columns = [f'ewm_{c}' for c in cols]
    return out

parts = []
for tid, grp in records.groupby('team_id'):
    grp = grp.sort_values('date').copy()
    r5 = roll(grp, ROLL, 5, 2)
    r10 = roll(grp, ROLL, 10, 5)
    e = ewm(grp, ROLL)
    grp['days_rest'] = grp['date'].diff().dt.days
    grp['is_home'] = (grp['venue']=='home').astype(float)
    block = pd.concat([
        grp[['fixture_id','team_id','date','venue','opp_id','target_slug','season',
             'is_home','days_rest','goals_for','goals_against']],
        r5, r10, e
    ], axis=1)
    parts.append(block)
team_features = pd.concat(parts).sort_values(['date','fixture_id']).reset_index(drop=True)
print(f"team_features: {team_features.shape}")

# Dixon-Coles per league (bounded + L2)

def dc_nll(p, df, n, l2=0.01):
    a=p[:n]; d=p[n:2*n]; ha=p[2*n]; rho=p[2*n+1]
    log_lh=np.clip(a[df['h_idx'].values]-d[df['a_idx'].values]+ha,-2,2)
    log_la=np.clip(a[df['a_idx'].values]-d[df['h_idx'].values],-2,2)
    lh=np.exp(log_lh); la=np.exp(log_la)
    hg=df['hg'].values; ag=df['ag'].values
    lg_h=np.array([lgamma(g+1) for g in hg])
    lg_a=np.array([lgamma(g+1) for g in ag])
    ll = hg*log_lh - lh - lg_h + ag*log_la - la - lg_a
    tau=np.ones(len(df))
    m00=(hg==0)&(ag==0); m01=(hg==0)&(ag==1)
    m10=(hg==1)&(ag==0); m11=(hg==1)&(ag==1)
    tau[m00]=np.maximum(1-lh[m00]*la[m00]*rho,1e-10)
    tau[m01]=np.maximum(1+lh[m01]*rho,1e-10)
    tau[m10]=np.maximum(1+la[m10]*rho,1e-10)
    tau[m11]=np.maximum(1-rho,1e-10)
    return -(ll+np.log(tau)).sum() + l2*(np.sum(a**2)+np.sum(d**2))

def fit_dc(matches):
    teams=sorted(set(matches['home_team_id'])|set(matches['away_team_id']))
    t2i={t:i for i,t in enumerate(teams)}; n=len(teams)
    df=pd.DataFrame({
        'h_idx':matches['home_team_id'].map(t2i).values,
        'a_idx':matches['away_team_id'].map(t2i).values,
        'hg':matches['home_score'].astype(int).values,
        'ag':matches['away_score'].astype(int).values,
    })
    x0=np.concatenate([np.zeros(n),np.zeros(n),[0.25,-0.05]])
    bnd=[(-2,2)]*n+[(-2,2)]*n+[(-1,1)]+[(-0.3,0.3)]
    res=minimize(dc_nll,x0,args=(df,n,0.01),method='L-BFGS-B',bounds=bnd,
                 options={'maxiter':500})
    a_c=res.x[:n]-res.x[:n].mean()
    d_c=res.x[n:2*n]-res.x[n:2*n].mean()
    return {'t2i':t2i,'a':a_c,'d':d_c,'ha':float(res.x[2*n]),'rho':float(res.x[2*n+1])}

def dc_pred(m, h, a):
    t2i=m['t2i']
    if h not in t2i or a not in t2i: return None,None
    hi,ai=t2i[h],t2i[a]
    lh=np.exp(np.clip(m['a'][hi]-m['d'][ai]+m['ha'],-2,2))
    la=np.exp(np.clip(m['a'][ai]-m['d'][hi],-2,2))
    return float(lh),float(la)

def dc_probs(lh, la, rho=0.0, max_g=8):
    p_h=poisson.pmf(np.arange(max_g+1),lh)
    p_a=poisson.pmf(np.arange(max_g+1),la)
    P=np.outer(p_h,p_a)
    P[0,0]*=max(1-lh*la*rho,1e-10); P[0,1]*=max(1+lh*rho,1e-10)
    P[1,0]*=max(1+la*rho,1e-10);    P[1,1]*=max(1-rho,1e-10)
    P/=P.sum()
    p_home=np.tril(P,-1).sum(); p_draw=np.diag(P).sum(); p_away=np.triu(P,1).sum()
    total=np.add.outer(np.arange(max_g+1),np.arange(max_g+1))
    return float(p_home),float(p_draw),float(p_away),float(P[total>=3].sum()),float(P[1:,1:].sum())

TRAIN_SEASONS = [2023, 2024]
print("\nFitting Dixon-Coles per league...")
dc_models = {}
fix_train = fixtures.dropna(subset=['home_score','away_score'])
for slug in fix_train['target_slug'].unique():
    sub = fix_train[(fix_train['target_slug']==slug) & (fix_train['season'].isin(TRAIN_SEASONS))]
    if len(sub) < 50: continue
    m = fit_dc(sub)
    dc_models[slug] = m
    print(f"  {slug:25s}  ha={m['ha']:+.3f}  rho={m['rho']:+.3f}")

dc_rows = []
for _, m in fixtures.iterrows():
    if m['target_slug'] not in dc_models: continue
    lh, la = dc_pred(dc_models[m['target_slug']], m['home_team_id'], m['away_team_id'])
    if lh is None: continue
    p_h, p_d, p_a, p_o, p_b = dc_probs(lh, la, rho=dc_models[m['target_slug']]['rho'])
    dc_rows.append({
        'fixture_id': m['fixture_id'],
        'dc_lambda_home': lh, 'dc_lambda_away': la,
        'dc_p_home': p_h, 'dc_p_draw': p_d, 'dc_p_away': p_a,
        'dc_p_over25': p_o, 'dc_p_btts': p_b,
        'dc_lambda_total': lh+la, 'dc_lambda_diff': lh-la,
    })
dc_df = pd.DataFrame(dc_rows)
dc_df.to_parquet(PROCESSED / 'dc_features.parquet', index=False)
print(f"dc_features: {dc_df.shape}")

# Also save the DC models — Stage 2 will need them for the base rate offset
with open(PROCESSED / 'dc_models.pkl', 'wb') as f:
    pickle.dump(dc_models, f)

INITIAL_ELO=1500; HOME_ADV=50; K_BASE=20
COMP_W = {'champions_league':1.10,'europa_league':1.05,'turkish_super_lig':0.95}

def expected(ra,rb,ha=0): return 1.0/(1.0+10**((rb-ra-ha)/400))
def gd_mult(gd):
    if abs(gd)<=1: return 1.0
    if abs(gd)==2: return 1.5
    return (11+abs(gd))/8.0

ft = fix_train[fix_train['season'].isin(TRAIN_SEASONS)].sort_values('date').reset_index(drop=True)
elo={}; hl=[]; al=[]
for _,m in ft.iterrows():
    h,a = m['home_team_id'], m['away_team_id']
    h_r,a_r = elo.get(h,INITIAL_ELO), elo.get(a,INITIAL_ELO)
    hl.append(h_r); al.append(a_r)
    e_h = expected(h_r,a_r,HOME_ADV)
    s_h = 1.0 if m['home_score']>m['away_score'] else (0.5 if m['home_score']==m['away_score'] else 0.0)
    k = K_BASE * gd_mult(m['home_score']-m['away_score']) * COMP_W.get(m['target_slug'],1.0)
    elo[h] = h_r + k*(s_h-e_h)
    elo[a] = a_r + k*((1.0-s_h)-(1.0-e_h))

f25 = fixtures[fixtures['season']==2025].sort_values('date').reset_index(drop=True)
hl25 = [elo.get(m['home_team_id'],INITIAL_ELO) for _,m in f25.iterrows()]
al25 = [elo.get(m['away_team_id'],INITIAL_ELO) for _,m in f25.iterrows()]

elo_features = pd.concat([
    pd.DataFrame({'fixture_id':ft['fixture_id'].values,'home_elo':hl,'away_elo':al}),
    pd.DataFrame({'fixture_id':f25['fixture_id'].values,'home_elo':hl25,'away_elo':al25}),
]).drop_duplicates('fixture_id')
elo_features['elo_diff'] = elo_features['home_elo'] - elo_features['away_elo']
elo_features['elo_expected_home'] = [expected(h,a,HOME_ADV) for h,a in zip(elo_features['home_elo'],elo_features['away_elo'])]
elo_features.to_parquet(PROCESSED / 'elo_features.parquet', index=False)
print(f"elo_features: {elo_features.shape}")

def standings(fix_in):
    rows=[]
    for (slug,season),grp in fix_in.groupby(['target_slug','season']):
        grp=grp.sort_values('date').reset_index(drop=True)
        pts={}; played={}; gd={}
        for _,m in grp.iterrows():
            h,a=m['home_team_id'],m['away_team_id']
            for t in [h,a]:
                pts.setdefault(t,0); played.setdefault(t,0); gd.setdefault(t,0)
            srt=sorted(pts.keys(),key=lambda t:(pts[t],gd[t]),reverse=True)
            h_pos=srt.index(h)+1 if played[h]>0 else np.nan
            a_pos=srt.index(a)+1 if played[a]>0 else np.nan
            rows.append({
                'fixture_id':m['fixture_id'],
                'home_pos':h_pos,'away_pos':a_pos,
                'home_pts':pts[h],'away_pts':pts[a],
                'home_played':played[h],'away_played':played[a],
                'home_ppg':pts[h]/max(played[h],1),'away_ppg':pts[a]/max(played[a],1),
            })
            if pd.notna(m['home_score']) and pd.notna(m['away_score']):
                hg,ag=int(m['home_score']),int(m['away_score'])
                played[h]+=1; played[a]+=1; gd[h]+=hg-ag; gd[a]+=ag-hg
                if hg>ag: pts[h]+=3
                elif hg<ag: pts[a]+=3
                else: pts[h]+=1; pts[a]+=1
    return pd.DataFrame(rows)

stand = standings(fixtures)
stand['pos_diff'] = stand['away_pos'] - stand['home_pos']
stand['ppg_diff'] = stand['home_ppg'] - stand['away_ppg']
stand.to_parquet(PROCESSED / 'standings_features.parquet', index=False)
print(f"standings: {stand.shape}")

# Pivot to match-level + GBM with locked test

home_tf = team_features[team_features['venue']=='home']
away_tf = team_features[team_features['venue']=='away']
meta = {'fixture_id','team_id','opp_id','date','venue','target_slug','season',
        'goals_for','goals_against','is_home'}
fcols = [c for c in team_features.columns if c not in meta]

home_f = home_tf[['fixture_id']+fcols].add_prefix('h_').rename(columns={'h_fixture_id':'fixture_id'})
away_f = away_tf[['fixture_id']+fcols].add_prefix('a_').rename(columns={'a_fixture_id':'fixture_id'})

mf = fixtures[['fixture_id','date','home_team_id','away_team_id',
               'home_team_name','away_team_name',
               'home_score','away_score','target_slug','season']].copy()
mf = mf.merge(home_f,on='fixture_id',how='left') \
       .merge(away_f,on='fixture_id',how='left') \
       .merge(dc_df,on='fixture_id',how='left') \
       .merge(elo_features,on='fixture_id',how='left') \
       .merge(stand,on='fixture_id',how='left')

for col in ['elo','pos','pts','ppg']:
    if f'home_{col}' in mf.columns and f'away_{col}' in mf.columns:
        mf[f'diff_{col}'] = mf[f'home_{col}'] - mf[f'away_{col}']

mf = mf.copy().replace([np.inf,-np.inf],np.nan)
mf['target'] = np.where(mf['home_score']>mf['away_score'],0,
                np.where(mf['home_score']<mf['away_score'],2,1))
mf['has_result'] = mf['home_score'].notna() & mf['away_score'].notna()
mf.to_parquet(PROCESSED / 'match_features_v2.parquet', index=False)
print(f"match_features_v2: {mf.shape}")

# Train/Val/Test split — last 30% of 2025 is LOCKED test
df = mf[mf['has_result']].copy().sort_values('date').reset_index(drop=True)
df_2025 = df[df['season']==2025].sort_values('date')
test_cutoff = df_2025.iloc[int(len(df_2025)*0.70)]['date']
train_val = df[df['date']<test_cutoff].copy()
test      = df[df['date']>=test_cutoff].copy()
print(f"\ntrain+val: {len(train_val)}, test: {len(test)}")

EXTRA = ['home_elo','away_elo','elo_diff','elo_expected_home',
         'home_pos','away_pos','home_pts','away_pts',
         'home_ppg','away_ppg','pos_diff','ppg_diff']
FEATS = [c for c in df.columns
         if (c.startswith(('h_','a_','diff_','dc_')) or c in EXTRA)
         and pd.api.types.is_numeric_dtype(df[c])
         and df[c].notna().sum()>0 and df[c].nunique(dropna=True)>1]

X_tv = train_val[FEATS].values.astype(np.float32)
y_tv = train_val['target'].values.astype(int)
X_test = test[FEATS].values.astype(np.float32)
y_test = test['target'].values.astype(int)

params = dict(n_estimators=400, learning_rate=0.04, max_depth=4,
              subsample=0.75, colsample_bytree=0.7, min_child_weight=5,
              gamma=1.0, reg_alpha=0.5, reg_lambda=2.0,
              objective='multi:softprob', num_class=3,
              eval_metric='mlogloss', random_state=42, n_jobs=-1)

es_split = int(len(X_tv)*0.80)
model = xgb.XGBClassifier(**params, early_stopping_rounds=30, verbosity=0)
model.fit(X_tv[:es_split], y_tv[:es_split],
          eval_set=[(X_tv[es_split:], y_tv[es_split:])], verbose=False)

test_proba = model.predict_proba(X_test)
n=len(y_test); oh=np.zeros((n,3)); oh[np.arange(n),y_test]=1
brier = float(((test_proba-oh)**2).sum(axis=1).mean())
ll = log_loss(y_test, test_proba)
print(f"\nTest Brier: {brier:.4f}, LogLoss: {ll:.4f}, n_features: {len(FEATS)}")

test_out = test[['fixture_id','date','target_slug',
                 'home_team_name','away_team_name',
                 'home_score','away_score','target']].copy()
test_out['p_home'] = test_proba[:,0]
test_out['p_draw'] = test_proba[:,1]
test_out['p_away'] = test_proba[:,2]
test_out.to_parquet(PROCESSED / 'test_predictions.parquet', index=False)

with open(PROCESSED / 'pre_match_model.pkl', 'wb') as f:
    pickle.dump({'model':model,'features':FEATS,'test_brier':brier,'test_logloss':ll}, f)

print(f"\nSaved test_predictions.parquet and pre_match_model.pkl")
print("Pre-match feature reconstruction complete.")

# State-space construction and hazard model


# State stream and scoring-hazard model

This section builds the minute-by-minute state stream and fits Poisson scoring-hazard models for home and away goal arrival.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import pickle
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import log_loss

PROCESSED = Path('/content/drive/MyDrive/soccer-betting/data/processed/api_football')

def log(msg, indent=0):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {'  '*indent}{msg}", flush=True)

events   = pd.read_parquet(PROCESSED / 'events_clean.parquet')
fixtures = pd.read_parquet(PROCESSED / 'fixtures_usable.parquet')
stats    = pd.read_parquet(PROCESSED / 'statistics.parquet')

fixtures['date'] = pd.to_datetime(fixtures['date'], utc=True)

log("STAGE 1: state extractor")
log(f"events: {events.shape}, fixtures: {fixtures.shape}, stats: {stats.shape}", indent=1)

team_stats = stats.set_index(['fixture_id', 'team_id'])[
    ['expected_goals', 'total_shots', 'shots_on_goal']
].copy()
team_stats['expected_goals'] = pd.to_numeric(team_stats['expected_goals'], errors='coerce')
team_stats['total_shots']    = pd.to_numeric(team_stats['total_shots'],    errors='coerce')
team_stats['shots_on_goal']  = pd.to_numeric(team_stats['shots_on_goal'],  errors='coerce')

SHOT_TYPES = {'Goal', 'Var'}  # We have these in event types

# For state extraction, the variables we CAN compute reliably from events:
#   - team xG (for the calibration, distributed proportionally to elapsed time)

# Stage 1: build state at every minute (1..90 + extra)

def goal_team_side(row, fix_lookup):
    """Return 'H' or 'A' for which team scored / was carded."""
    fid = row['fixture_id']
    fx = fix_lookup.get(fid)
    if fx is None:
        return None
    return 'H' if row['team_id'] == fx['home_team_id'] else 'A'

fix_lookup = fixtures.set_index('fixture_id')[
    ['home_team_id', 'away_team_id', 'home_score', 'away_score', 'target_slug', 'season']
].to_dict('index')

events_by_fid = events.groupby('fixture_id')

log("building state rows for each fixture...", indent=1)
state_rows = []
fid_list = sorted(fix_lookup.keys())
n_fix = len(fid_list)

for i, fid in enumerate(fid_list):
    if fid not in events_by_fid.groups:
        continue
    fx = fix_lookup[fid]
    h_id = fx['home_team_id']
    a_id = fx['away_team_id']

    ev = events_by_fid.get_group(fid)

    h_xg_total = team_stats.loc[(fid, h_id), 'expected_goals'] if (fid, h_id) in team_stats.index else 0
    a_xg_total = team_stats.loc[(fid, a_id), 'expected_goals'] if (fid, a_id) in team_stats.index else 0
    h_xg_total = float(h_xg_total) if pd.notna(h_xg_total) else 0.0
    a_xg_total = float(a_xg_total) if pd.notna(a_xg_total) else 0.0

    ev = ev.sort_values(['minute', 'extra'], na_position='first').reset_index(drop=True)
    ev['side'] = ev['team_id'].apply(lambda t: 'H' if t == h_id else ('A' if t == a_id else None))

    max_min = int(min(ev['minute'].max() if len(ev) > 0 else 90, 95))
    end_min = max(90, max_min)

    # Initialize state
    h_score = 0; a_score = 0
    h_red = 0;   a_red = 0

    ev_by_min = ev.groupby('minute')

    for minute in range(0, end_min + 1):
        # State BEFORE events at this minute resolve
        # (so a goal at minute=23 means state at minute=23 has score from before)
        # Actually: we want pre-state at time t for predicting goal in (t, t+1].
        # So emit state, then process events.

        h_xg_cum = h_xg_total * (minute / 90.0)
        a_xg_cum = a_xg_total * (minute / 90.0)

        state_rows.append({
            'fixture_id':  fid,
            'minute':      minute,
            'h_score':     h_score,
            'a_score':     a_score,
            'score_diff':  h_score - a_score,  # H - A
            'h_red':       h_red,
            'a_red':       a_red,
            'red_diff':    h_red - a_red,
            'h_xg_cum':    h_xg_cum,
            'a_xg_cum':    a_xg_cum,
            'minutes_left': max(end_min - minute, 0),
        })

        if minute in ev_by_min.groups:
            for _, e in ev_by_min.get_group(minute).iterrows():
                if e['side'] not in ('H', 'A'):
                    continue
                if e['type'] == 'Goal':
                    if e['side'] == 'H': h_score += 1
                    else:                a_score += 1
                elif e['type'] == 'Card' and 'Red' in str(e['detail']):
                    if e['side'] == 'H': h_red += 1
                    else:                a_red += 1

    if (i + 1) % 1000 == 0:
        log(f"  {i+1}/{n_fix} fixtures processed", indent=1)

state = pd.DataFrame(state_rows)
log(f"state rows: {state.shape}", indent=1)

log("\nbuilding goal-arrival labels...", indent=1)

goals = events[events['type'] == 'Goal'].copy()
goals['side'] = goals.apply(
    lambda r: 'H' if r['team_id'] == fix_lookup.get(r['fixture_id'], {}).get('home_team_id') else 'A',
    axis=1
)
goal_minutes = goals.groupby(['fixture_id', 'minute', 'side']).size().reset_index(name='n_goals')

state['h_goal_next'] = 0
state['a_goal_next'] = 0

# A goal event at minute=t corresponds to label for state at minute=t-1
# (state at t-1 predicts what happens in (t-1, t])
for _, g in goal_minutes.iterrows():
    fid = g['fixture_id']
    pred_min = max(g['minute'] - 1, 0)
    mask = (state['fixture_id'] == fid) & (state['minute'] == pred_min)
    if g['side'] == 'H':
        state.loc[mask, 'h_goal_next'] += g['n_goals']
    else:
        state.loc[mask, 'a_goal_next'] += g['n_goals']

# Drop states from minute 90+ where we have no label info beyond
state_fit = state[state['minute'] < 90].copy()

log(f"fit set: {state_fit.shape}", indent=1)
log(f"  total H goals: {state_fit['h_goal_next'].sum()}", indent=2)
log(f"  total A goals: {state_fit['a_goal_next'].sum()}", indent=2)
log(f"  base goal rate per minute: "
    f"H={state_fit['h_goal_next'].mean():.4f}  A={state_fit['a_goal_next'].mean():.4f}", indent=2)

state.to_parquet(PROCESSED / 'state_stream.parquet', index=False)
log(f"saved -> state_stream.parquet", indent=1)

# STAGE 2: Hazard model (Poisson regression)

log("STAGE 2: hazard model (Poisson regression)")

state_fit['minute_norm'] = state_fit['minute'] / 90.0
state_fit['late_game']   = (state_fit['minute'] >= 75).astype(int)
state_fit['early_game']  = (state_fit['minute'] <= 15).astype(int)

state_fit['h_trailing'] = (state_fit['score_diff'] < 0).astype(int)
state_fit['a_trailing'] = (state_fit['score_diff'] > 0).astype(int)

# Features for home goal hazard
HAZARD_FEATS = [
    'minute_norm', 'late_game', 'early_game',
    'score_diff',
    'red_diff',
    'h_xg_cum', 'a_xg_cum',
    'h_trailing', 'a_trailing',
]

# Held-out validation: split by fixture, not by row (no leakage)
np.random.seed(42)
all_fids = state_fit['fixture_id'].unique()
np.random.shuffle(all_fids)
n_train = int(len(all_fids) * 0.8)
train_fids = set(all_fids[:n_train])
val_fids   = set(all_fids[n_train:])

train_mask = state_fit['fixture_id'].isin(train_fids)
val_mask   = state_fit['fixture_id'].isin(val_fids)
log(f"train fixtures: {len(train_fids)}, val fixtures: {len(val_fids)}", indent=1)
log(f"train rows: {train_mask.sum()}, val rows: {val_mask.sum()}", indent=1)

X_train = state_fit.loc[train_mask, HAZARD_FEATS].values
X_val   = state_fit.loc[val_mask,   HAZARD_FEATS].values
yh_train = state_fit.loc[train_mask, 'h_goal_next'].values
yh_val   = state_fit.loc[val_mask,   'h_goal_next'].values
ya_train = state_fit.loc[train_mask, 'a_goal_next'].values
ya_val   = state_fit.loc[val_mask,   'a_goal_next'].values

log("\nfitting home-goal hazard...", indent=1)
hazard_h = PoissonRegressor(alpha=1.0, max_iter=300)
hazard_h.fit(X_train, yh_train)

log("fitting away-goal hazard...", indent=1)
hazard_a = PoissonRegressor(alpha=1.0, max_iter=300)
hazard_a.fit(X_train, ya_train)

def eval_hazard(model, X, y, name):
    lam = model.predict(X)
    lam = np.clip(lam, 1e-6, 0.5)  # numerical safety
    # Poisson log-likelihood on val
    ll = (y * np.log(lam) - lam).mean()
    base_rate = y.mean()
    base_lam = np.full_like(y, base_rate, dtype=float)
    ll_base = (y * np.log(base_lam) - base_lam).mean()
    log(f"{name}: held-out logL/row = {ll:.5f}  (baseline {ll_base:.5f}, "
        f"improvement {ll - ll_base:+.5f})", indent=2)
    return lam

lam_h_val = eval_hazard(hazard_h, X_val, yh_val, "home hazard")
lam_a_val = eval_hazard(hazard_a, X_val, ya_val, "away hazard")

log("\nhome hazard coefficients:", indent=1)
for f, c in zip(HAZARD_FEATS, hazard_h.coef_):
    log(f"  {f:18s}  {c:+.4f}", indent=2)
log(f"  intercept           {hazard_h.intercept_:+.4f}", indent=2)

log("\naway hazard coefficients:", indent=1)
for f, c in zip(HAZARD_FEATS, hazard_a.coef_):
    log(f"  {f:18s}  {c:+.4f}", indent=2)
log(f"  intercept           {hazard_a.intercept_:+.4f}", indent=2)

# Sanity plot: fitted hazard by minute (should rise late)

log("\nfitted hazard by minute bucket (sanity check):", indent=1)
state_fit['lam_h_pred'] = hazard_h.predict(state_fit[HAZARD_FEATS].values)
state_fit['lam_a_pred'] = hazard_a.predict(state_fit[HAZARD_FEATS].values)
buckets = pd.cut(state_fit['minute'], bins=[0, 15, 30, 45, 60, 75, 90], include_lowest=True)
summary = state_fit.groupby(buckets, observed=True).agg(
    actual_h=('h_goal_next', 'mean'),
    fitted_h=('lam_h_pred', 'mean'),
    actual_a=('a_goal_next', 'mean'),
    fitted_a=('lam_a_pred', 'mean'),
).round(4)
print(summary.to_string())

with open(PROCESSED / 'hazard_models.pkl', 'wb') as f:
    pickle.dump({
        'hazard_h': hazard_h,
        'hazard_a': hazard_a,
        'features': HAZARD_FEATS,
        'val_fids': list(val_fids),
        'train_fids': list(train_fids),
    }, f)
log(f"\nsaved -> hazard_models.pkl", indent=1)

log("STAGES 1+2 COMPLETE")
log("=" * 70)

# Hazard-model refinement

This section refits the hazard model using regularization, corrected xG features, and team-strength controls.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import pickle
from sklearn.linear_model import PoissonRegressor

PROCESSED = Path('/content/drive/MyDrive/soccer-betting/data/processed/api_football')

def log(msg, indent=0):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {'  '*indent}{msg}", flush=True)

# Load existing state stream + DC features for team strength

state = pd.read_parquet(PROCESSED / 'state_stream.parquet')
fixtures = pd.read_parquet(PROCESSED / 'fixtures_usable.parquet')
dc_df = pd.read_parquet(PROCESSED / 'dc_features.parquet')
events = pd.read_parquet(PROCESSED / 'events_clean.parquet')

log(f"state: {state.shape}, dc: {dc_df.shape}", indent=1)

# Re-compute goal labels (state.parquet may not have them)
fix_lookup = fixtures.set_index('fixture_id')[
    ['home_team_id', 'away_team_id']
].to_dict('index')

goals = events[events['type'] == 'Goal'].copy()
goals['side'] = goals.apply(
    lambda r: 'H' if r['team_id'] == fix_lookup.get(r['fixture_id'], {}).get('home_team_id') else 'A',
    axis=1
)
goal_minutes = goals.groupby(['fixture_id', 'minute', 'side']).size().reset_index(name='n_goals')

if 'h_goal_next' not in state.columns:
    state['h_goal_next'] = 0
    state['a_goal_next'] = 0
    for _, g in goal_minutes.iterrows():
        fid = g['fixture_id']; pred_min = max(g['minute'] - 1, 0)
        mask = (state['fixture_id'] == fid) & (state['minute'] == pred_min)
        if g['side'] == 'H':
            state.loc[mask, 'h_goal_next'] += g['n_goals']
        else:
            state.loc[mask, 'a_goal_next'] += g['n_goals']

state = state[state['minute'] < 90].copy()

state = state.merge(
    dc_df[['fixture_id', 'dc_lambda_home', 'dc_lambda_away']],
    on='fixture_id', how='left'
)

state = state.dropna(subset=['dc_lambda_home', 'dc_lambda_away'])
log(f"after merging DC: {state.shape}", indent=1)

# This lets the model learn arbitrary time shape without imposing linearity
for lo, hi in [(0, 15), (15, 30), (30, 45), (45, 60), (60, 75), (75, 90)]:
    state[f'min_{lo}_{hi}'] = ((state['minute'] >= lo) & (state['minute'] < hi)).astype(int)

# Game-state features
state['h_leading']  = (state['score_diff'] > 0).astype(int)
state['a_leading']  = (state['score_diff'] < 0).astype(int)
state['big_lead']   = (state['score_diff'].abs() >= 2).astype(int)
state['h_man_up']   = (state['red_diff'] < 0).astype(int)   # home has more men (away has more reds)
state['a_man_up']   = (state['red_diff'] > 0).astype(int)

state['log_dc_lh'] = np.log(state['dc_lambda_home'].clip(lower=0.05))
state['log_dc_la'] = np.log(state['dc_lambda_away'].clip(lower=0.05))

HAZARD_FEATS = [
    'min_0_15', 'min_15_30', 'min_30_45', 'min_45_60', 'min_60_75', 'min_75_90',
    'h_leading', 'a_leading', 'big_lead',
    'h_man_up', 'a_man_up',
    'log_dc_lh', 'log_dc_la',
]
HAZARD_FEATS = [f for f in HAZARD_FEATS if f != 'min_45_60']  # 45-60 = reference bin

np.random.seed(42)
all_fids = state['fixture_id'].unique()
np.random.shuffle(all_fids)
n_train = int(len(all_fids) * 0.8)
train_fids = set(all_fids[:n_train]); val_fids = set(all_fids[n_train:])

train_mask = state['fixture_id'].isin(train_fids)
val_mask   = state['fixture_id'].isin(val_fids)

X_train = state.loc[train_mask, HAZARD_FEATS].values
X_val   = state.loc[val_mask,   HAZARD_FEATS].values
yh_train = state.loc[train_mask, 'h_goal_next'].values
yh_val   = state.loc[val_mask,   'h_goal_next'].values
ya_train = state.loc[train_mask, 'a_goal_next'].values
ya_val   = state.loc[val_mask,   'a_goal_next'].values

log(f"train: {X_train.shape}, val: {X_val.shape}", indent=1)

log("\nfitting hazards with alpha=1e-4...", indent=1)
hazard_h = PoissonRegressor(alpha=1e-4, max_iter=500)
hazard_h.fit(X_train, yh_train)

hazard_a = PoissonRegressor(alpha=1e-4, max_iter=500)
hazard_a.fit(X_train, ya_train)

log("\nhome hazard coefficients:", indent=1)
for f, c in zip(HAZARD_FEATS, hazard_h.coef_):
    log(f"  {f:14s}  {c:+.4f}", indent=2)
log(f"  intercept       {hazard_h.intercept_:+.4f}", indent=2)

log("\naway hazard coefficients:", indent=1)
for f, c in zip(HAZARD_FEATS, hazard_a.coef_):
    log(f"  {f:14s}  {c:+.4f}", indent=2)
log(f"  intercept       {hazard_a.intercept_:+.4f}", indent=2)

state['lam_h_pred'] = hazard_h.predict(state[HAZARD_FEATS].values)
state['lam_a_pred'] = hazard_a.predict(state[HAZARD_FEATS].values)
buckets = pd.cut(state['minute'], bins=[0, 15, 30, 45, 60, 75, 90], include_lowest=True)
summary = state.groupby(buckets, observed=True).agg(
    actual_h=('h_goal_next', 'mean'), fitted_h=('lam_h_pred', 'mean'),
    actual_a=('a_goal_next', 'mean'), fitted_a=('lam_a_pred', 'mean'),
).round(4)
log("\nfit by minute bucket (should track):", indent=1)
print(summary.to_string())

log("\nfit by lead state (sanity check):", indent=1)
behav = state.groupby(['h_leading', 'a_leading'], observed=True).agg(
    actual_h=('h_goal_next', 'mean'), fitted_h=('lam_h_pred', 'mean'),
    actual_a=('a_goal_next', 'mean'), fitted_a=('lam_a_pred', 'mean'),
).round(4)
print(behav.to_string())

log("\nfit by red-card state:", indent=1)
red_check = state.groupby(['h_man_up', 'a_man_up'], observed=True).agg(
    actual_h=('h_goal_next', 'mean'), fitted_h=('lam_h_pred', 'mean'),
    actual_a=('a_goal_next', 'mean'), fitted_a=('lam_a_pred', 'mean'),
    n=('h_goal_next', 'size'),
).round(4)
print(red_check.to_string())

with open(PROCESSED / 'hazard_models.pkl', 'wb') as f:
    pickle.dump({
        'hazard_h': hazard_h, 'hazard_a': hazard_a,
        'features': HAZARD_FEATS,
        'val_fids': list(val_fids), 'train_fids': list(train_fids),
    }, f)
log(f"\nsaved -> hazard_models.pkl", indent=1)
log("\nSTAGE 2 v2 COMPLETE")

# Simulation calibration

This section calibrates fixture-level simulation multipliers for home and away scoring intensity.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import pickle
import time

PROCESSED = Path('/content/drive/MyDrive/soccer-betting/data/processed/api_football')

def log(msg, indent=0):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {'  '*indent}{msg}", flush=True)

with open(PROCESSED / 'hazard_models.pkl', 'rb') as f:
    haz = pickle.load(f)
hazard_h = haz['hazard_h']; hazard_a = haz['hazard_a']
HAZARD_FEATS = haz['features']
F_IDX = {f: i for i, f in enumerate(HAZARD_FEATS)}

with open(PROCESSED / 'pre_match_model.pkl', 'rb') as f:
    pmm = pickle.load(f)
gbm_features = pmm['features']
gbm_model    = pmm['model']

mf = pd.read_parquet(PROCESSED / 'match_features_v2.parquet')

mf_fit = mf[mf['has_result']].copy()
X_all = mf_fit[gbm_features].values.astype(np.float32)
gbm_proba = gbm_model.predict_proba(X_all)
mf_fit['gbm_p_home'] = gbm_proba[:, 0]
mf_fit['gbm_p_draw'] = gbm_proba[:, 1]
mf_fit['gbm_p_away'] = gbm_proba[:, 2]

cal_data = mf_fit[['fixture_id', 'dc_lambda_home', 'dc_lambda_away',
                   'gbm_p_home', 'gbm_p_draw', 'gbm_p_away']].dropna().reset_index(drop=True)
log(f"fixtures to calibrate: {len(cal_data)}", indent=1)

# Pre-extract hazard coefficients into named scalars/arrays

beta_h = hazard_h.coef_.astype(np.float32)
beta_a = hazard_a.coef_.astype(np.float32)
b0_h   = float(hazard_h.intercept_)
b0_a   = float(hazard_a.intercept_)

bh_h_lead  = beta_h[F_IDX['h_leading']]
bh_a_lead  = beta_h[F_IDX['a_leading']]
bh_big     = beta_h[F_IDX['big_lead']]
ba_h_lead  = beta_a[F_IDX['h_leading']]
ba_a_lead  = beta_a[F_IDX['a_leading']]
ba_big     = beta_a[F_IDX['big_lead']]

# Score-INDEPENDENT log-hazard contribution: depends only on
# For calibration: man_up=0 always (neutral pre-match state).

def time_bin_log_contrib(minute):
    """log-contribution from time-bin features alone (scalar)."""
    contrib_h = 0.0; contrib_a = 0.0
    if minute < 15:
        contrib_h = beta_h[F_IDX['min_0_15']];  contrib_a = beta_a[F_IDX['min_0_15']]
    elif minute < 30:
        contrib_h = beta_h[F_IDX['min_15_30']]; contrib_a = beta_a[F_IDX['min_15_30']]
    elif minute < 45:
        contrib_h = beta_h[F_IDX['min_30_45']]; contrib_a = beta_a[F_IDX['min_30_45']]
    elif minute < 60:
        pass  # reference
    elif minute < 75:
        contrib_h = beta_h[F_IDX['min_60_75']]; contrib_a = beta_a[F_IDX['min_60_75']]
    else:
        contrib_h = beta_h[F_IDX['min_75_90']]; contrib_a = beta_a[F_IDX['min_75_90']]
    return contrib_h, contrib_a

time_log_h = np.array([time_bin_log_contrib(m)[0] for m in range(90)], dtype=np.float32)
time_log_a = np.array([time_bin_log_contrib(m)[1] for m in range(90)], dtype=np.float32)

def simulate_batch(log_lh_arr, log_la_arr, alpha_h, alpha_a,
                   n_sims=1500, h_score0=0, a_score0=0, h_red0=0, a_red0=0,
                   start_minute=0, end_minute=90, rng=None):
    """
    Simulate n_sims paths for each of len(log_lh_arr) fixtures, in parallel.

    Returns:
        p_home_win, p_draw, p_away_win: arrays of shape (n_fixtures,)

    Tensor shape during simulation: (n_fix, n_sims).
    """
    if rng is None:
        rng = np.random.default_rng(42)
    n_fix = len(log_lh_arr)

    log_lh = log_lh_arr.astype(np.float32).reshape(-1, 1)  # (n_fix, 1)
    log_la = log_la_arr.astype(np.float32).reshape(-1, 1)

    # Static log-hazard contribution (no time, no score dep) per fixture
    static_log_h = (b0_h
                    + beta_h[F_IDX['log_dc_lh']] * log_lh
                    + beta_h[F_IDX['log_dc_la']] * log_la)  # shape (n_fix, 1)
    static_log_a = (b0_a
                    + beta_a[F_IDX['log_dc_lh']] * log_lh
                    + beta_a[F_IDX['log_dc_la']] * log_la)

    log_alpha_h = np.log(alpha_h, dtype=np.float32)
    log_alpha_a = np.log(alpha_a, dtype=np.float32)

    h_score = np.full((n_fix, n_sims), h_score0, dtype=np.int16)
    a_score = np.full((n_fix, n_sims), a_score0, dtype=np.int16)

    for minute in range(start_minute, end_minute):
        t_log_h = time_log_h[minute]
        t_log_a = time_log_a[minute]

        score_diff = h_score - a_score  # (n_fix, n_sims)
        h_lead = (score_diff > 0).astype(np.float32)
        a_lead = (score_diff < 0).astype(np.float32)
        big    = (np.abs(score_diff) >= 2).astype(np.float32)

        delta_h = bh_h_lead * h_lead + bh_a_lead * a_lead + bh_big * big
        delta_a = ba_h_lead * h_lead + ba_a_lead * a_lead + ba_big * big

        # Final log-hazard, then exponentiate
        log_lam_h = static_log_h + t_log_h + delta_h + log_alpha_h
        log_lam_a = static_log_a + t_log_a + delta_a + log_alpha_a

        lam_h = np.exp(log_lam_h)
        lam_a = np.exp(log_lam_a)

        # Sample Poisson goals -- vectorized across (n_fix, n_sims)
        gh = rng.poisson(lam_h).astype(np.int16)
        ga = rng.poisson(lam_a).astype(np.int16)

        h_score += gh
        a_score += ga

    p_h = (h_score > a_score).mean(axis=1)
    p_d = (h_score == a_score).mean(axis=1)
    p_a = (h_score < a_score).mean(axis=1)
    return p_h, p_d, p_a

# Vectorized calibration

def calibrate_batch(batch_df, n_sims=1500, seed=42):
    """
    Calibrate (alpha_h, alpha_a) for every fixture in batch_df via grid search.

    Returns DataFrame with columns: fixture_id, alpha_h, alpha_a, kl
    """
    log_lh = np.log(batch_df['dc_lambda_home'].clip(lower=0.05).values)
    log_la = np.log(batch_df['dc_lambda_away'].clip(lower=0.05).values)
    target = batch_df[['gbm_p_home', 'gbm_p_draw', 'gbm_p_away']].values  # (n_fix, 3)

    n_fix = len(batch_df)
    best_kl    = np.full(n_fix, np.inf)
    best_alpha_h = np.full(n_fix, 1.0)
    best_alpha_a = np.full(n_fix, 1.0)

    coarse = [0.6, 0.8, 1.0, 1.2, 1.4]

    for ah in coarse:
        for aa in coarse:
            rng = np.random.default_rng(seed)  # same seed across (ah,aa) for fairness
            p_h, p_d, p_a = simulate_batch(log_lh, log_la, ah, aa,
                                             n_sims=n_sims, rng=rng)
            sim_p = np.stack([p_h, p_d, p_a], axis=1)  # (n_fix, 3)
            sim_p = np.clip(sim_p, 1e-9, 1)
            tgt   = np.clip(target, 1e-9, 1)
            kl = (tgt * np.log(tgt / sim_p)).sum(axis=1)  # (n_fix,)

            mask = kl < best_kl
            best_kl[mask] = kl[mask]
            best_alpha_h[mask] = ah
            best_alpha_a[mask] = aa

    pairs = list(zip(best_alpha_h, best_alpha_a))
    from collections import Counter
    most_common_ah, most_common_aa = Counter(pairs).most_common(1)[0][0]

    fine_ah = np.linspace(max(most_common_ah - 0.15, 0.3), most_common_ah + 0.15, 7)
    fine_aa = np.linspace(max(most_common_aa - 0.15, 0.3), most_common_aa + 0.15, 7)

    for ah in fine_ah:
        for aa in fine_aa:
            rng = np.random.default_rng(seed)
            p_h, p_d, p_a = simulate_batch(log_lh, log_la, ah, aa,
                                             n_sims=n_sims, rng=rng)
            sim_p = np.stack([p_h, p_d, p_a], axis=1)
            sim_p = np.clip(sim_p, 1e-9, 1)
            tgt   = np.clip(target, 1e-9, 1)
            kl = (tgt * np.log(tgt / sim_p)).sum(axis=1)

            mask = kl < best_kl
            best_kl[mask] = kl[mask]
            best_alpha_h[mask] = ah
            best_alpha_a[mask] = aa

    return pd.DataFrame({
        'fixture_id': batch_df['fixture_id'].values,
        'alpha_h': best_alpha_h,
        'alpha_a': best_alpha_a,
        'kl': best_kl,
    })

log("VECTORIZED CALIBRATION")

BATCH_SIZE = 200  # 200 fixtures × 1500 sims = 300k paths per simulate call

# Quick timing test on one batch
log(f"\ntiming one batch of {BATCH_SIZE}...", indent=1)
t0 = time.time()
test_batch = cal_data.head(BATCH_SIZE).reset_index(drop=True)
test_result = calibrate_batch(test_batch, n_sims=1500, seed=42)
batch_time = time.time() - t0
log(f"  {batch_time:.1f}s for {BATCH_SIZE} fixtures ({batch_time/BATCH_SIZE*1000:.0f}ms/fixture)", indent=2)

n_batches = int(np.ceil(len(cal_data) / BATCH_SIZE))
total_eta = batch_time * n_batches / 60
log(f"  full ETA: ~{total_eta:.1f} min for {len(cal_data)} fixtures", indent=2)

if total_eta > 30:
    log(f"\nETA still too long. Dropping n_sims to 1000.", indent=1)
    N_SIMS = 1000
else:
    N_SIMS = 1500

log(f"\nrunning full calibration with n_sims={N_SIMS}, batch={BATCH_SIZE}...", indent=1)
all_results = []
t0 = time.time()
for i in range(0, len(cal_data), BATCH_SIZE):
    batch = cal_data.iloc[i:i+BATCH_SIZE].reset_index(drop=True)
    r = calibrate_batch(batch, n_sims=N_SIMS, seed=42 + i)
    all_results.append(r)
    n_done = i + len(batch)
    if (i // BATCH_SIZE + 1) % 5 == 0 or n_done >= len(cal_data):
        elapsed = time.time() - t0
        rate = n_done / elapsed
        eta = (len(cal_data) - n_done) / max(rate, 0.1) / 60
        log(f"  [{n_done}/{len(cal_data)}] elapsed={elapsed:.0f}s eta={eta:.1f}min", indent=2)

cal_df = pd.concat(all_results, ignore_index=True)
total_elapsed = (time.time() - t0) / 60
log(f"\nfull calibration done in {total_elapsed:.1f} min", indent=1)

log("\ncalibration distribution:", indent=1)
log(f"  alpha_h:  mean={cal_df['alpha_h'].mean():.3f}  std={cal_df['alpha_h'].std():.3f}  "
    f"range=[{cal_df['alpha_h'].min():.2f}, {cal_df['alpha_h'].max():.2f}]", indent=2)
log(f"  alpha_a:  mean={cal_df['alpha_a'].mean():.3f}  std={cal_df['alpha_a'].std():.3f}  "
    f"range=[{cal_df['alpha_a'].min():.2f}, {cal_df['alpha_a'].max():.2f}]", indent=2)
log(f"  kl:       mean={cal_df['kl'].mean():.4f}  median={cal_df['kl'].median():.4f}  "
    f"max={cal_df['kl'].max():.4f}", indent=2)
log(f"  kl > 0.01: {(cal_df['kl'] > 0.01).sum()} fixtures "
    f"({(cal_df['kl'] > 0.01).mean()*100:.1f}%)", indent=2)
log(f"  kl > 0.05: {(cal_df['kl'] > 0.05).sum()} fixtures "
    f"({(cal_df['kl'] > 0.05).mean()*100:.1f}%)", indent=2)

edge = ((cal_df['alpha_h'] <= 0.45) | (cal_df['alpha_h'] >= 1.55) |
        (cal_df['alpha_a'] <= 0.45) | (cal_df['alpha_a'] >= 1.55))
log(f"  hit grid edges: {edge.sum()} fixtures ({edge.mean()*100:.1f}%)", indent=2)

cal_df.to_parquet(PROCESSED / 'sim_calibration.parquet', index=False)
log(f"\nsaved -> sim_calibration.parquet", indent=1)
log("STAGE 4 v2 COMPLETE")
log("=" * 70)

# Forward simulation and conditional state markets


# Conditional in-play market simulator

This section simulates match continuations from each observed state and converts them into conditional in-play market probabilities.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import pickle
from datetime import datetime

PROCESSED = Path('/content/drive/MyDrive/soccer-betting/data/processed/api_football')
FID = 1515514  # Gala 5-2 Juve, UCL R16, 2026-02-17

def log(msg, indent=0):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {'  '*indent}{msg}", flush=True)

with open(PROCESSED / 'hazard_models.pkl', 'rb') as f:
    haz = pickle.load(f)
hazard_h = haz['hazard_h']; hazard_a = haz['hazard_a']
HAZARD_FEATS = haz['features']
F_IDX = {f: i for i, f in enumerate(HAZARD_FEATS)}
beta_h = hazard_h.coef_; b0_h = hazard_h.intercept_
beta_a = hazard_a.coef_; b0_a = hazard_a.intercept_

events   = pd.read_parquet(PROCESSED / 'events_clean.parquet')
fixtures = pd.read_parquet(PROCESSED / 'fixtures_usable.parquet')
mf       = pd.read_parquet(PROCESSED / 'match_features_v2.parquet')

try:
    cal_df = pd.read_parquet(PROCESSED / 'sim_calibration.parquet')
    cal_row = cal_df[cal_df['fixture_id'] == FID]
    if len(cal_row) > 0:
        ALPHA_H = float(cal_row.iloc[0]['alpha_h'])
        ALPHA_A = float(cal_row.iloc[0]['alpha_a'])
    else:
        ALPHA_H, ALPHA_A = 1.17, 1.17
except FileNotFoundError:
    ALPHA_H, ALPHA_A = 1.17, 1.17

log(f"using alphas: alpha_h={ALPHA_H:.3f}, alpha_a={ALPHA_A:.3f}")

def simulate_full(start_minute, h_score0, a_score0, h_red0, a_red0,
                  dc_lh, dc_la, alpha_h=1.0, alpha_a=1.0,
                  n_sims=4000, end_minute=90, seed=42):
    """
    Returns:
        h_score_final:  (n_sims,) final home score
        a_score_final:  (n_sims,) final away score
        h_score_at_45:  (n_sims,) home score at minute 45 (or start_minute, if later)
        a_score_at_45:  (n_sims,)
        next_goal_side: (n_sims,) 'H' or 'A' or 'N' (no more goals)
        next_goal_minute: (n_sims,) minute of next goal after start_minute, or 999 if none
    """
    rng = np.random.default_rng(seed)
    log_lh = np.log(max(dc_lh, 0.05)); log_la = np.log(max(dc_la, 0.05))

    h_score = np.full(n_sims, h_score0, dtype=np.int32)
    a_score = np.full(n_sims, a_score0, dtype=np.int32)
    h_score_at_45 = np.full(n_sims, h_score0, dtype=np.int32)
    a_score_at_45 = np.full(n_sims, a_score0, dtype=np.int32)

    next_goal_side = np.full(n_sims, 'N', dtype='<U1')
    next_goal_minute = np.full(n_sims, 999, dtype=np.int16)
    next_goal_recorded = np.zeros(n_sims, dtype=bool)

    h_man_up_v = float(a_red0 > h_red0)
    a_man_up_v = float(h_red0 > a_red0)

    for minute in range(start_minute, end_minute):
        time_feats = np.zeros(len(HAZARD_FEATS), dtype=np.float32)
        if   minute < 15: time_feats[F_IDX['min_0_15']]  = 1
        elif minute < 30: time_feats[F_IDX['min_15_30']] = 1
        elif minute < 45: time_feats[F_IDX['min_30_45']] = 1
        elif minute < 60: pass
        elif minute < 75: time_feats[F_IDX['min_60_75']] = 1
        else:             time_feats[F_IDX['min_75_90']] = 1
        time_feats[F_IDX['h_man_up']]  = h_man_up_v
        time_feats[F_IDX['a_man_up']]  = a_man_up_v
        time_feats[F_IDX['log_dc_lh']] = log_lh
        time_feats[F_IDX['log_dc_la']] = log_la

        base_log_h = b0_h + np.dot(beta_h, time_feats)
        base_log_a = b0_a + np.dot(beta_a, time_feats)

        score_diff = h_score - a_score
        h_lead = (score_diff > 0).astype(np.float32)
        a_lead = (score_diff < 0).astype(np.float32)
        big    = (np.abs(score_diff) >= 2).astype(np.float32)

        delta_h = (beta_h[F_IDX['h_leading']] * h_lead
                 + beta_h[F_IDX['a_leading']] * a_lead
                 + beta_h[F_IDX['big_lead']]  * big)
        delta_a = (beta_a[F_IDX['h_leading']] * h_lead
                 + beta_a[F_IDX['a_leading']] * a_lead
                 + beta_a[F_IDX['big_lead']]  * big)

        lam_h = np.exp(base_log_h + delta_h) * alpha_h
        lam_a = np.exp(base_log_a + delta_a) * alpha_a

        gh = rng.poisson(lam_h).astype(np.int32)
        ga = rng.poisson(lam_a).astype(np.int32)

        scored_h_now = (gh > 0) & ~next_goal_recorded
        scored_a_now = (ga > 0) & ~next_goal_recorded
        h_only = scored_h_now & ~scored_a_now
        a_only = scored_a_now & ~scored_h_now
        both   = scored_h_now & scored_a_now

        next_goal_side[h_only] = 'H'
        next_goal_side[a_only] = 'A'
        if both.any():
            h_wins_tiebreak = rng.random(both.sum()) < (lam_h[both] / (lam_h[both] + lam_a[both]))
            both_idx = np.where(both)[0]
            for i, idx in enumerate(both_idx):
                next_goal_side[idx] = 'H' if h_wins_tiebreak[i] else 'A'

        next_goal_minute[scored_h_now | scored_a_now] = minute
        next_goal_recorded[scored_h_now | scored_a_now] = True

        h_score += gh
        a_score += ga

        if minute == 44:  # we just finished minute 44, so end-of-44 = start-of-45
            h_score_at_45 = h_score.copy()
            a_score_at_45 = a_score.copy()

    return {
        'h_final': h_score, 'a_final': a_score,
        'h_at_45': h_score_at_45, 'a_at_45': a_score_at_45,
        'next_goal_side': next_goal_side, 'next_goal_minute': next_goal_minute,
    }

def compute_markets(sim_result, start_minute, h_score0, a_score0):
    """Aggregate sim_result into market probabilities."""
    hf = sim_result['h_final']; af = sim_result['a_final']
    h45 = sim_result['h_at_45']; a45 = sim_result['a_at_45']

    p_h = (hf > af).mean()
    p_d = (hf == af).mean()
    p_a = (hf < af).mean()
    total = hf + af
    p_o15 = (total >= 2).mean()
    p_o25 = (total >= 3).mean()
    p_o35 = (total >= 4).mean()
    p_btts = ((hf >= 1) & (af >= 1)).mean()
    p_h_by2 = ((hf - af) >= 2).mean()
    p_h_2plus = (hf >= 2).mean()
    p_a_2plus = (af >= 2).mean()

    if start_minute < 45:
        h1_h_goals = h45 - h_score0
        h1_a_goals = a45 - a_score0
        p_h1_h = (h1_h_goals > h1_a_goals).mean()
        p_h1_d = (h1_h_goals == h1_a_goals).mean()
        p_h1_a = (h1_h_goals < h1_a_goals).mean()
    else:
        p_h1_h = p_h1_d = p_h1_a = np.nan

    h2_h_goals = hf - h45
    h2_a_goals = af - a45
    p_h2_h = (h2_h_goals > h2_a_goals).mean()
    p_h2_d = (h2_h_goals == h2_a_goals).mean()
    p_h2_a = (h2_h_goals < h2_a_goals).mean()

    next_side = sim_result['next_goal_side']
    p_next_h = (next_side == 'H').mean()
    p_next_a = (next_side == 'A').mean()
    p_no_more_goals = (next_side == 'N').mean()

    next_min = sim_result['next_goal_minute']
    p_goal_in_7 = ((next_min - start_minute <= 7) & (next_side != 'N')).mean()
    p_goal_in_15 = ((next_min - start_minute <= 15) & (next_side != 'N')).mean()

    return {
        'p_home_win': float(p_h), 'p_draw': float(p_d), 'p_away_win': float(p_a),
        'p_over15': float(p_o15), 'p_over25': float(p_o25), 'p_over35': float(p_o35),
        'p_btts': float(p_btts),
        'p_home_by2plus': float(p_h_by2),
        'p_home_2plus': float(p_h_2plus), 'p_away_2plus': float(p_a_2plus),
        'p_h1_home': float(p_h1_h), 'p_h1_draw': float(p_h1_d), 'p_h1_away': float(p_h1_a),
        'p_h2_home': float(p_h2_h), 'p_h2_draw': float(p_h2_d), 'p_h2_away': float(p_h2_a),
        'p_next_home': float(p_next_h), 'p_next_away': float(p_next_a),
        'p_no_more_goals': float(p_no_more_goals),
        'p_goal_in_7': float(p_goal_in_7), 'p_goal_in_15': float(p_goal_in_15),
    }

fx = fixtures[fixtures['fixture_id'] == FID].iloc[0]
mf_row = mf[mf['fixture_id'] == FID].iloc[0]
home_id, away_id = fx['home_team_id'], fx['away_team_id']
home_name, away_name = fx['home_team_name'], fx['away_team_name']
final_h, final_a = int(fx['home_score']), int(fx['away_score'])
dc_lh, dc_la = mf_row['dc_lambda_home'], mf_row['dc_lambda_away']

print(f"\n{'='*80}")
print(f"MATCH: {home_name} {final_h}-{final_a} {away_name}")
print(f"{'='*80}")

m_events = events[events['fixture_id'] == FID].sort_values(['minute', 'extra']).copy()
m_events['side'] = m_events['team_id'].apply(lambda t: 'H' if t == home_id else 'A')

goals = m_events[m_events['type'] == 'Goal'][['minute', 'side', 'player_name']].values.tolist()
reds = m_events[(m_events['type'] == 'Card')
                & (m_events['detail'].fillna('').str.contains('Red'))][
    ['minute', 'side', 'player_name']].values.tolist()

# Walk through every minute, compute all 12+ markets

log("\nrunning per-minute simulation across all 90 minutes (4000 sims each)...")

results = []
for minute in range(0, 91):
    h_sc = a_sc = h_rd = a_rd = 0
    for _, e in m_events.iterrows():
        if e['minute'] >= minute:
            break
        if e['side'] == 'H':
            if e['type'] == 'Goal': h_sc += 1
            elif e['type'] == 'Card' and 'Red' in str(e['detail']): h_rd += 1
        elif e['side'] == 'A':
            if e['type'] == 'Goal': a_sc += 1
            elif e['type'] == 'Card' and 'Red' in str(e['detail']): a_rd += 1

    sim = simulate_full(minute, h_sc, a_sc, h_rd, a_rd,
                        dc_lh, dc_la, alpha_h=ALPHA_H, alpha_a=ALPHA_A,
                        n_sims=4000)
    markets = compute_markets(sim, minute, h_sc, a_sc)
    markets.update({'min': minute, 'h_sc': h_sc, 'a_sc': a_sc,
                    'h_rd': h_rd, 'a_rd': a_rd})
    results.append(markets)

traj = pd.DataFrame(results)
log(f"trajectory: {len(traj)} time points, {len(traj.columns)} columns")

print(f"\n{'='*150}")
print(f"ALL MARKETS AT KEY MOMENTS")
print(f"{'='*150}")
header = (f"{'min':>3}  {'sc':>4}  "
          f"{'P(H)':>5} {'P(D)':>5} {'P(A)':>5}  "
          f"{'O1.5':>5} {'O2.5':>5} {'O3.5':>5}  "
          f"{'BTTS':>5} {'H+2':>5} {'A+2':>5}  "
          f"{'H1H':>5} {'H1D':>5} {'H1A':>5}  "
          f"{'H2H':>5} {'H2D':>5} {'H2A':>5}  "
          f"{'NxH':>5} {'NxA':>5} {'g<7':>5} "
          f" event")
print(header)
print('-' * 150)

key_minutes = sorted(set([0, 15, 30, 45, 60, 75, 89]
                         + [g[0] for g in goals] + [g[0] + 1 for g in goals]
                         + [r[0] for r in reds] + [r[0] + 1 for r in reds]))
for km in key_minutes:
    if km > 90: continue
    row = traj[traj['min'] == km]
    if len(row) == 0: continue
    r = row.iloc[0]

    evt = ""
    for gm, gs, gp in goals:
        if gm == km:
            team = 'H' if gs == 'H' else 'A'
            evt += f"⚽{team}({(gp or '?')[:8]}) "
    for rm, rs, _ in reds:
        if rm == km:
            evt += f"🟥{rs} "

    def fmt(v):
        if pd.isna(v): return f"{'–':>5}"
        return f"{v:.3f}"

    print(f"{int(r['min']):>3}  "
          f"{int(r['h_sc'])}-{int(r['a_sc']):<2}  "
          f"{fmt(r['p_home_win'])} {fmt(r['p_draw'])} {fmt(r['p_away_win'])}  "
          f"{fmt(r['p_over15'])} {fmt(r['p_over25'])} {fmt(r['p_over35'])}  "
          f"{fmt(r['p_btts'])} {fmt(r['p_home_2plus'])} {fmt(r['p_away_2plus'])}  "
          f"{fmt(r['p_h1_home'])} {fmt(r['p_h1_draw'])} {fmt(r['p_h1_away'])}  "
          f"{fmt(r['p_h2_home'])} {fmt(r['p_h2_draw'])} {fmt(r['p_h2_away'])}  "
          f"{fmt(r['p_next_home'])} {fmt(r['p_next_away'])} {fmt(r['p_goal_in_7'])} "
          f" {evt}")

print(f"\n{'='*80}")
print(f"CONDITIONAL PROP SHOWCASE: red card at minute 67 (Juventus)")
print(f"{'='*80}")
print(f"At minute 67, score is 3-2 (Galatasaray leading), Juventus just got a red card.")
print(f"Question: what's P(Galatasaray scores within 7 min)?")
print(f"          what's P(Juventus scores within 7 min)?")
print()

sim67 = simulate_full(67, 3, 2, 0, 1, dc_lh, dc_la,
                      alpha_h=ALPHA_H, alpha_a=ALPHA_A, n_sims=10000)
m67 = compute_markets(sim67, 67, 3, 2)

# Counterfactual: same state but no red card
sim67_nr = simulate_full(67, 3, 2, 0, 0, dc_lh, dc_la,
                          alpha_h=ALPHA_H, alpha_a=ALPHA_A, n_sims=10000)
m67_nr = compute_markets(sim67_nr, 67, 3, 2)

print(f"  WITH red card on Juventus:")
print(f"    P(next goal = Gala 11-man side):    {m67['p_next_home']:.3f}")
print(f"    P(next goal = Juve 10-man side):    {m67['p_next_away']:.3f}")
print(f"    P(any goal within 7 min):           {m67['p_goal_in_7']:.3f}")
print(f"    P(any goal within 15 min):          {m67['p_goal_in_15']:.3f}")
print(f"    P(home wins from here):             {m67['p_home_win']:.3f}")
print()
print(f"  WITHOUT red card (counterfactual):")
print(f"    P(next goal = Galatasaray):         {m67_nr['p_next_home']:.3f}")
print(f"    P(next goal = Juventus):            {m67_nr['p_next_away']:.3f}")
print(f"    P(any goal within 7 min):           {m67_nr['p_goal_in_7']:.3f}")
print(f"    P(any goal within 15 min):          {m67_nr['p_goal_in_15']:.3f}")
print(f"    P(home wins from here):             {m67_nr['p_home_win']:.3f}")
print()
print(f"  EFFECT of the red card on key probabilities:")
print(f"    P(next goal Gala) shift:    {m67['p_next_home'] - m67_nr['p_next_home']:+.3f}")
print(f"    P(next goal Juve) shift:    {m67['p_next_away'] - m67_nr['p_next_away']:+.3f}")
print(f"    P(home wins) shift:         {m67['p_home_win'] - m67_nr['p_home_win']:+.3f}")

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True,
                         gridspec_kw={'height_ratios': [3, 2, 2, 1.5]})

GALA_COLOR = '#FFA500'
JUVE_COLOR = '#000000'

ax = axes[0]
ax.plot(traj['min'], traj['p_home_win'], color=GALA_COLOR, lw=2.5, label='P(Gala)')
ax.plot(traj['min'], traj['p_draw'],     color='#888', lw=2.0, ls='--', label='P(Draw)')
ax.plot(traj['min'], traj['p_away_win'], color=JUVE_COLOR, lw=2.5, label='P(Juve)')
for minute, side, player in goals:
    color = GALA_COLOR if side == 'H' else JUVE_COLOR
    ax.axvline(minute, color=color, alpha=0.25, lw=1)
    ax.scatter([minute], [1.04], marker='v', s=120, color=color, zorder=5, clip_on=False)
for minute, side, _ in reds:
    color = GALA_COLOR if side == 'H' else JUVE_COLOR
    ax.axvline(minute, color=color, alpha=0.4, lw=1, ls=':')
    ax.scatter([minute], [-0.04], marker='s', s=80, color=color, zorder=5, clip_on=False)
ax.set_ylabel('P(1X2)', fontsize=11); ax.set_ylim(0, 1.05)
ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_title(f'{home_name} {final_h}-{final_a} {away_name}    UCL R16 1st leg, 2026-02-17',
             loc='left', fontsize=12)

# Panel 2: Total goals markets
ax = axes[1]
ax.plot(traj['min'], traj['p_over15'], color='#1f77b4', lw=1.8, label='P(O 1.5)')
ax.plot(traj['min'], traj['p_over25'], color='#ff7f0e', lw=1.8, label='P(O 2.5)')
ax.plot(traj['min'], traj['p_over35'], color='#d62728', lw=1.8, label='P(O 3.5)')
ax.plot(traj['min'], traj['p_btts'],   color='#2ca02c', lw=1.8, label='P(BTTS)')
ax.set_ylabel('Goal markets', fontsize=11); ax.set_ylim(0, 1.05)
ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(traj['min'], traj['p_h2_home'], color=GALA_COLOR, lw=1.8, label='P(H2: Gala wins H2)')
ax.plot(traj['min'], traj['p_h2_draw'], color='#888', lw=1.5, ls='--', label='P(H2: Draw)')
ax.plot(traj['min'], traj['p_h2_away'], color=JUVE_COLOR, lw=1.8, label='P(H2: Juve wins H2)')
ax.axvline(45, color='red', alpha=0.3, lw=1.5, ls='--')
ax.text(45.5, 0.95, 'HT', fontsize=9, color='red')
ax.set_ylabel('Second half 1X2', fontsize=11); ax.set_ylim(0, 1.05)
ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 4: Next-goal markets
ax = axes[3]
ax.plot(traj['min'], traj['p_next_home'], color=GALA_COLOR, lw=1.8, label='P(next goal: Gala)')
ax.plot(traj['min'], traj['p_next_away'], color=JUVE_COLOR, lw=1.8, label='P(next goal: Juve)')
ax.plot(traj['min'], traj['p_goal_in_7'], color='#9467bd', lw=1.5, ls=':', label='P(goal in next 7m)')
ax.set_ylabel('Next-goal market', fontsize=11)
ax.set_xlabel('Minute', fontsize=11)
ax.set_xlim(0, 90); ax.set_ylim(0, 1.05)
ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
out_path = PROCESSED / f'all_markets_{FID}.png'
plt.savefig(out_path, dpi=130, bbox_inches='tight')
plt.show()
log(f"\nsaved -> {out_path.name}")

traj.to_parquet(PROCESSED / f'markets_traj_{FID}.parquet', index=False)
log(f"saved -> markets_traj_{FID}.parquet")

print("\nDone.")

# State-space trigger validation


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import beta as beta_dist

PROCESSED = Path('/content/drive/MyDrive/soccer-betting/data/processed/api_football')
fixtures = pd.read_parquet(PROCESSED / 'fixtures_usable.parquet')
events   = pd.read_parquet(PROCESSED / 'events_clean.parquet')
mf       = pd.read_parquet(PROCESSED / 'match_features_v2.parquet')

fixtures['date'] = pd.to_datetime(fixtures['date'], utc=True)
mf['date'] = pd.to_datetime(mf['date'], utc=True)

fix = fixtures.merge(mf[['fixture_id', 'dc_p_home', 'dc_p_draw', 'dc_p_away']],
                     on='fixture_id', how='inner')
fix = fix[fix['home_score'].notna() & fix['away_score'].notna()].copy()
fix['favorite'] = np.where(fix['dc_p_home'] > fix['dc_p_away'], 'H', 'A')
fix['favorite_prob'] = fix[['dc_p_home', 'dc_p_away']].max(axis=1)
fix['big_favorite'] = fix['favorite_prob'] > 0.55

def build_match_state(fid, fix_row, m_events):
    home_id = fix_row['home_team_id']
    favorite_side = fix_row['favorite']
    home_score = int(fix_row['home_score'])
    away_score = int(fix_row['away_score'])
    total = home_score + away_score
    h_won = home_score > away_score
    a_won = away_score > home_score
    draw  = home_score == away_score

    fav_won  = (favorite_side == 'H' and h_won) or (favorite_side == 'A' and a_won)
    fav_drew = draw
    fav_avoid_loss = fav_won or fav_drew

    btts = (home_score >= 1) and (away_score >= 1)
    over15 = total >= 2
    over25 = total >= 3
    over35 = total >= 4

    # event-derived state
    goals = m_events[m_events['type'] == 'Goal'].sort_values(['minute', 'extra'])
    reds = m_events[(m_events['type'] == 'Card') &
                    (m_events['detail'].fillna('').str.contains('Red'))].sort_values(['minute', 'extra'])

    first_goal_min = None
    first_goal_side = None
    if len(goals) > 0:
        g0 = goals.iloc[0]
        first_goal_min = int(g0['minute'])
        first_goal_side = 'H' if g0['team_id'] == home_id else 'A'

    fav_conceded_first = (first_goal_side is not None
                          and first_goal_side != favorite_side)

    red_events = []
    for _, r in reds.iterrows():
        side = 'H' if r['team_id'] == home_id else 'A'
        red_events.append((int(r['minute']), side))

    # next goal after a given event - useful for trigger validation
    def next_goal_after(t):
        after = goals[goals['minute'] > t]
        if len(after) == 0:
            return None
        g = after.iloc[0]
        return 'H' if g['team_id'] == home_id else 'A'

    return {
        'fixture_id': fid,
        'league': fix_row['target_slug'],
        'season': fix_row['season'],
        'favorite_side': favorite_side,
        'favorite_prob': fix_row['favorite_prob'],
        'big_favorite': fix_row['big_favorite'],
        'home_score': home_score, 'away_score': away_score,
        'first_goal_min': first_goal_min, 'first_goal_side': first_goal_side,
        'fav_conceded_first': fav_conceded_first,
        'red_events': red_events,
        'fav_won': fav_won, 'fav_drew': fav_drew, 'fav_avoid_loss': fav_avoid_loss,
        'btts': btts, 'over15': over15, 'over25': over25, 'over35': over35,
        'next_goal_after_first': next_goal_after(first_goal_min) if first_goal_min is not None else None,
    }

events_by_fid = events.groupby('fixture_id')
states = []
for fid, fix_row in fix.set_index('fixture_id').iterrows():
    if fid not in events_by_fid.groups:
        continue
    m_ev = events_by_fid.get_group(fid)
    states.append(build_match_state(fid, fix_row, m_ev))

S = pd.DataFrame(states)
print(f"matches with full state: {len(S)}")

def wilson_ci(x, n, conf=0.95):
    """Wilson 95% interval for a binomial proportion. Better than normal-approx for small n."""
    if n == 0:
        return (0, 0)
    z = 1.96 if conf == 0.95 else 2.576
    p = x / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = z * np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / denom
    return (center - half, center + half)

def evaluate_trigger(df, mask, label, bets):
    """Given a boolean trigger mask, compute hit rate + CI for each bet column.
    bets: list of (display_name, df_column)."""
    sub = df[mask]
    n = len(sub)
    out = {'trigger': label, 'n_signals': n}
    for bet_name, col in bets:
        hits = int(sub[col].sum())
        rate = hits / n if n > 0 else 0
        lo, hi = wilson_ci(hits, n)
        out[f'{bet_name}_rate'] = rate
        out[f'{bet_name}_ci'] = (lo, hi)
        out[f'{bet_name}_breakeven_odds'] = (1/rate) if rate > 0 else float('inf')
    return out

bets_to_test = [
    ('BTTS',      'btts'),
    ('Over 1.5',  'over15'),
    ('Over 2.5',  'over25'),
    ('Over 3.5',  'over35'),
    ('Fav avoid loss', 'fav_avoid_loss'),
    ('Fav win',   'fav_won'),
]

S['fav_conceded_first_early'] = S['fav_conceded_first'] & (S['first_goal_min'].fillna(999) <= 35)
S['fav_conceded_first_any'] = S['fav_conceded_first']
S['big_fav_conceded_first'] = S['big_favorite'] & S['fav_conceded_first']
S['big_fav_conceded_first_early'] = S['big_favorite'] & S['fav_conceded_first_early']

# baseline: just "early goal" (any team)
S['any_goal_before_35'] = S['first_goal_min'].fillna(999) <= 35

# baselines first
baselines = []
baselines.append(evaluate_trigger(S, pd.Series([True]*len(S)), 'ALL MATCHES (baseline)', bets_to_test))
baselines.append(evaluate_trigger(S, S['any_goal_before_35'], 'Any team scored before min 35', bets_to_test))
baselines.append(evaluate_trigger(S, S['big_favorite'], 'Big favorite playing (p > 0.55)', bets_to_test))

triggers = []
triggers.append(evaluate_trigger(S, S['fav_conceded_first'], 'Favorite conceded first', bets_to_test))
triggers.append(evaluate_trigger(S, S['fav_conceded_first_early'], 'Favorite conceded first before min 35', bets_to_test))
triggers.append(evaluate_trigger(S, S['big_fav_conceded_first_early'], 'BIG fav conceded first before min 35', bets_to_test))

def print_results(rows, title):
    print('\n' + '='*100)
    print(title)
    print('='*100)
    print(f"{'trigger':50s} {'n':>5s}  {'BTTS':>14s} {'Over 1.5':>14s} {'Over 2.5':>14s} {'Fav avoid':>14s}")
    for r in rows:
        def fmt(name):
            rate = r[f'{name}_rate']
            lo, hi = r[f'{name}_ci']
            return f"{rate*100:>5.1f}% [{lo*100:.0f}-{hi*100:.0f}]"
        print(f"{r['trigger']:50s} {r['n_signals']:>5d}  "
              f"{fmt('BTTS'):>14s} {fmt('Over 1.5'):>14s} "
              f"{fmt('Over 2.5'):>14s} {fmt('Fav avoid loss'):>14s}")

print_results(baselines, 'BASELINES (for comparison)')
print_results(triggers, 'STATE-SPACE TRIGGER RULES')

print('\n' + '='*100)
print('BREAK-EVEN PRICES (max market price at which the bet is +EV, no fees)')
print('='*100)
print(f"{'trigger':50s} {'bet':>15s}  {'hit_rate':>10s} {'breakeven_odds':>15s} {'breakeven_price':>16s}")
for r in triggers:
    for bet_name, _ in bets_to_test:
        rate = r[f'{bet_name}_rate']
        if rate == 0: continue
        be_odds = 1 / rate
        be_price_cents = rate * 100
        print(f"{r['trigger']:50s} {bet_name:>15s}  {rate*100:>9.1f}% {be_odds:>15.2f} {be_price_cents:>15.1f}c")

S_save = S.copy()
S_save['red_events'] = S_save['red_events'].apply(lambda x: str(x) if isinstance(x, list) else x)
S_save.to_parquet(PROCESSED / 'trigger_states.parquet', index=False)
print(f"saved -> trigger_states.parquet")

# Synthetic live-odds backtest and stress tests


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import poisson
from pathlib import Path

PROCESSED = Path('/content/drive/MyDrive/soccer-betting/data/processed/api_football')

bt = pd.read_parquet(PROCESSED / 'backtest_results.parquet')
mf = pd.read_parquet(PROCESSED / 'match_features_v2.parquet')

dc_lookup = mf.set_index('fixture_id')[['dc_lambda_home', 'dc_lambda_away']].to_dict('index')

def dc_state_probs(h_sc, a_sc, t_minutes_left, dc_lh, dc_la, max_extra=10):
    frac = max(t_minutes_left, 0) / 90.0
    lam_h = max(dc_lh * frac, 0)
    lam_a = max(dc_la * frac, 0)

    if lam_h < 1e-9 and lam_a < 1e-9:
        if h_sc > a_sc:
            return 1.0, 0.0, 0.0
        if h_sc < a_sc:
            return 0.0, 0.0, 1.0
        return 0.0, 1.0, 0.0

    h_extra = np.arange(max_extra + 1)
    a_extra = np.arange(max_extra + 1)

    p_h_extra = poisson.pmf(h_extra, lam_h)
    p_a_extra = poisson.pmf(a_extra, lam_a)

    M = np.outer(p_h_extra, p_a_extra)

    final_h = h_sc + h_extra.reshape(-1, 1)
    final_a = a_sc + a_extra.reshape(1, -1)

    p_home = M[final_h > final_a].sum()
    p_draw = M[final_h == final_a].sum()
    p_away = M[final_h < final_a].sum()

    Z = p_home + p_draw + p_away
    if Z <= 0:
        return np.nan, np.nan, np.nan

    return p_home / Z, p_draw / Z, p_away / Z

dc_probs = []

for _, r in bt.iterrows():
    fid = r['fixture_id']

    if fid not in dc_lookup:
        dc_probs.append((np.nan, np.nan, np.nan))
        continue

    lh = dc_lookup[fid]['dc_lambda_home']
    la = dc_lookup[fid]['dc_lambda_away']

    minutes_left = 90 - r['checkpoint']

    p_h, p_d, p_a = dc_state_probs(
        r['h_sc'],
        r['a_sc'],
        minutes_left,
        lh,
        la
    )

    dc_probs.append((p_h, p_d, p_a))

bt[['dc_p_home', 'dc_p_draw', 'dc_p_away']] = dc_probs

def add_state_flags(bt):
    bt = bt.sort_values(['fixture_id', 'checkpoint']).copy()

    bt['score_total'] = bt['h_sc'] + bt['a_sc']
    bt['score_diff'] = bt['h_sc'] - bt['a_sc']
    bt['abs_score_diff'] = bt['score_diff'].abs()
    bt['has_goal'] = bt['score_total'] > 0

    bt['prev_h_sc'] = bt.groupby('fixture_id')['h_sc'].shift(1)
    bt['prev_a_sc'] = bt.groupby('fixture_id')['a_sc'].shift(1)

    bt['score_changed_since_prev_checkpoint'] = (
        (bt['prev_h_sc'].notna()) &
        ((bt['h_sc'] != bt['prev_h_sc']) | (bt['a_sc'] != bt['prev_a_sc']))
    )

    red_cols = [c for c in bt.columns if 'red' in c.lower()]
    if len(red_cols) > 0:
        bt['red_card_state_proxy'] = bt[red_cols].fillna(0).sum(axis=1)
        bt['prev_red_card_state_proxy'] = bt.groupby('fixture_id')['red_card_state_proxy'].shift(1)
        bt['red_changed_since_prev_checkpoint'] = (
            bt['prev_red_card_state_proxy'].notna() &
            (bt['red_card_state_proxy'] != bt['prev_red_card_state_proxy'])
        )
    else:
        bt['red_card_state_proxy'] = 0
        bt['red_changed_since_prev_checkpoint'] = False

    bt['state_changed_since_prev_checkpoint'] = (
        bt['score_changed_since_prev_checkpoint'] |
        bt['red_changed_since_prev_checkpoint']
    )

    bt['closing_favorite'] = np.select(
        [
            (bt['market_p_home'] > bt['market_p_draw']) & (bt['market_p_home'] > bt['market_p_away']),
            (bt['market_p_away'] > bt['market_p_home']) & (bt['market_p_away'] > bt['market_p_draw'])
        ],
        ['home', 'away'],
        default='draw'
    )

    return bt

bt = add_state_flags(bt)

def normalize_3way(p_h, p_d, p_a, eps=1e-8):
    arr = np.array([p_h, p_d, p_a], dtype=float)
    arr = np.clip(arr, eps, 1 - eps)
    arr = arr / arr.sum()
    return arr[0], arr[1], arr[2]

def state_dependent_weight(row, mode='conservative'):
    t = float(row['checkpoint'])
    abs_score_diff = float(row.get('abs_score_diff', 0))
    has_goal = bool(row.get('has_goal', False))
    state_changed = bool(row.get('state_changed_since_prev_checkpoint', False))
    red_state = float(row.get('red_card_state_proxy', 0))

    if mode == 'loose':
        w0 = 0.75
        time_decay = np.exp(-0.010 * t)
        score_decay = np.exp(-0.35 * abs_score_diff)
        w = w0 * time_decay * score_decay

        if has_goal:
            w = min(w, 0.45)
        if state_changed:
            w = min(w, 0.35)
        if red_state > 0:
            w = min(w, 0.30)
        if t >= 70:
            w = min(w, 0.25)
        if t >= 80:
            w = min(w, 0.18)

        return float(np.clip(w, 0.05, 0.80))

    if mode == 'base':
        w0 = 0.65
        time_decay = np.exp(-0.015 * t)
        score_decay = np.exp(-0.55 * abs_score_diff)
        w = w0 * time_decay * score_decay

        if has_goal:
            w = min(w, 0.35)
        if state_changed:
            w = min(w, 0.25)
        if red_state > 0:
            w = min(w, 0.20)
        if t >= 70:
            w = min(w, 0.15)
        if t >= 80:
            w = min(w, 0.10)

        return float(np.clip(w, 0.02, 0.70))

    if mode == 'brutal':
        w0 = 0.50
        time_decay = np.exp(-0.025 * t)
        score_decay = np.exp(-0.80 * abs_score_diff)
        w = w0 * time_decay * score_decay

        if has_goal:
            w = min(w, 0.22)
        if state_changed:
            w = min(w, 0.12)
        if red_state > 0:
            w = min(w, 0.10)
        if t >= 60:
            w = min(w, 0.10)
        if t >= 70:
            w = min(w, 0.06)
        if t >= 80:
            w = min(w, 0.03)

        return float(np.clip(w, 0.00, 0.55))

    raise ValueError("mode must be one of: loose, base, brutal")

def synthetic_market_probs(row, weight_mode='base'):
    w = state_dependent_weight(row, mode=weight_mode)

    mp_h = w * row['market_p_home'] + (1 - w) * row['dc_p_home']
    mp_d = w * row['market_p_draw'] + (1 - w) * row['dc_p_draw']
    mp_a = w * row['market_p_away'] + (1 - w) * row['dc_p_away']

    mp_h, mp_d, mp_a = normalize_3way(mp_h, mp_d, mp_a)

    return mp_h, mp_d, mp_a, w

def get_model_probs(row, source):
    if source == 'model':
        return row['model_p_home'], row['model_p_draw'], row['model_p_away']

    if source == 'dc':
        return row['dc_p_home'], row['dc_p_draw'], row['dc_p_away']

    if source == 'closing':
        return row['market_p_home'], row['market_p_draw'], row['market_p_away']

    if source == 'random':
        return 1 / 3, 1 / 3, 1 / 3

    raise ValueError("source must be one of: model, dc, closing, random")

def actual_outcome(row):
    if row['final_h'] > row['final_a']:
        return 'H'
    if row['final_h'] < row['final_a']:
        return 'A'
    return 'D'

def run_strategy(
    bt,
    label,
    prob_source='model',
    weight_mode='base',
    overround=1.07,
    slippage_prob=0.01,
    edge_threshold=0.05,
    kelly_frac=0.10,
    min_checkpoint=0,
    max_checkpoint=85,
    ban_same_checkpoint_after_state_change=True,
    max_stake=0.02,
    verbose=True
):
    bets = []

    needed = [
        'dc_p_home', 'dc_p_draw', 'dc_p_away',
        'market_p_home', 'market_p_draw', 'market_p_away',
        'final_h', 'final_a'
    ]

    tmp = bt.dropna(subset=needed).copy()

    tmp = tmp[
        (tmp['checkpoint'] >= min_checkpoint) &
        (tmp['checkpoint'] <= max_checkpoint)
    ]

    for _, r in tmp.iterrows():
        if ban_same_checkpoint_after_state_change and bool(r.get('state_changed_since_prev_checkpoint', False)):
            continue

        sm_h, sm_d, sm_a, w_used = synthetic_market_probs(r, weight_mode=weight_mode)
        p_h, p_d, p_a = get_model_probs(r, prob_source)

        p_h, p_d, p_a = normalize_3way(p_h, p_d, p_a)

        market_probs = {
            'home': sm_h,
            'draw': sm_d,
            'away': sm_a
        }

        model_probs = {
            'home': p_h,
            'draw': p_d,
            'away': p_a
        }

        letters = {
            'home': 'H',
            'draw': 'D',
            'away': 'A'
        }

        out = actual_outcome(r)

        for side in ['home', 'draw', 'away']:
            p_model = model_probs[side]
            p_market_raw = market_probs[side]

            p_market = p_market_raw * overround + slippage_prob
            p_market = min(max(p_market, 1e-6), 0.999)

            edge = p_model - p_market

            if edge < edge_threshold:
                continue

            odds = 1.0 / p_market
            b = odds - 1.0
            q = 1.0 - p_model

            if b <= 0:
                continue

            full_kelly = (b * p_model - q) / b

            if full_kelly <= 0:
                continue

            stake = kelly_frac * full_kelly
            stake = min(stake, max_stake)

            if stake <= 0:
                continue

            won = letters[side] == out
            pnl = (odds - 1.0) * stake if won else -stake

            bets.append({
                'fixture_id': r['fixture_id'],
                'checkpoint': r['checkpoint'],
                'side': side,
                'prob_source': prob_source,
                'weight_mode': weight_mode,
                'w_used': w_used,
                'p_model': p_model,
                'p_market_raw': p_market_raw,
                'p_market_after_costs': p_market,
                'edge': edge,
                'odds': odds,
                'stake': stake,
                'won': won,
                'pnl': pnl,
                'score_home': r['h_sc'],
                'score_away': r['a_sc'],
                'score_diff': r['score_diff'],
                'has_goal': r['has_goal'],
                'state_changed_since_prev_checkpoint': r['state_changed_since_prev_checkpoint'],
                'actual_outcome': out,
            })

    bdf = pd.DataFrame(bets)

    if len(bdf) == 0:
        if verbose:
            print(f"{label}: no bets")
        return bdf

    stake_sum = bdf['stake'].sum()
    pnl_sum = bdf['pnl'].sum()
    roi = pnl_sum / stake_sum * 100 if stake_sum > 0 else np.nan

    if verbose:
        print(
            f"{label}: "
            f"n_bets={len(bdf)}  "
            f"hit_rate={bdf['won'].mean()*100:.1f}%  "
            f"ROI={roi:.2f}%  "
            f"stake={stake_sum:.2f}u  "
            f"pnl={pnl_sum:+.2f}u  "
            f"avg_edge={bdf['edge'].mean()*100:.2f}pp  "
            f"avg_w={bdf['w_used'].mean():.3f}"
        )

    return bdf

def summarize_by_checkpoint(bdf):
    if bdf is None or len(bdf) == 0:
        return pd.DataFrame()

    cp = bdf.groupby('checkpoint').agg(
        n=('fixture_id', 'count'),
        hit=('won', 'mean'),
        stake=('stake', 'sum'),
        pnl=('pnl', 'sum'),
        avg_edge=('edge', 'mean'),
        avg_w=('w_used', 'mean')
    )

    cp['roi_pct'] = cp['pnl'] / cp['stake'] * 100
    return cp.round(4)

def summarize_by_side(bdf):
    if bdf is None or len(bdf) == 0:
        return pd.DataFrame()

    s = bdf.groupby('side').agg(
        n=('fixture_id', 'count'),
        hit=('won', 'mean'),
        stake=('stake', 'sum'),
        pnl=('pnl', 'sum'),
        avg_edge=('edge', 'mean'),
        avg_odds=('odds', 'mean')
    )

    s['roi_pct'] = s['pnl'] / s['stake'] * 100
    return s.round(4)

def summarize_by_score_state(bdf):
    if bdf is None or len(bdf) == 0:
        return pd.DataFrame()

    tmp = bdf.copy()
    tmp['score_state'] = tmp['score_home'].astype(str) + '-' + tmp['score_away'].astype(str)

    s = tmp.groupby('score_state').agg(
        n=('fixture_id', 'count'),
        hit=('won', 'mean'),
        stake=('stake', 'sum'),
        pnl=('pnl', 'sum'),
        avg_edge=('edge', 'mean'),
        avg_w=('w_used', 'mean')
    ).sort_values('n', ascending=False)

    s['roi_pct'] = s['pnl'] / s['stake'] * 100
    return s.round(4)

print("Synthetic live odds stress-test harness")
print("Changes vs old version:")
print("  1. state-dependent closing-weight decay")
print("  2. post-goal and late-game reactivity caps")
print("  3. larger overround/slippage options")
print("  4. no same-checkpoint betting after score/red-card changes")
print("  5. DC-only and closing-only dumb baselines")
print()

configs = [
    {
        'label': 'MODEL loose',
        'prob_source': 'model',
        'weight_mode': 'loose',
        'overround': 1.05,
        'slippage_prob': 0.005,
        'edge_threshold': 0.03,
        'kelly_frac': 0.25,
        'max_stake': 0.05,
    },
    {
        'label': 'MODEL base conservative',
        'prob_source': 'model',
        'weight_mode': 'base',
        'overround': 1.07,
        'slippage_prob': 0.010,
        'edge_threshold': 0.05,
        'kelly_frac': 0.10,
        'max_stake': 0.02,
    },
    {
        'label': 'MODEL brutal',
        'prob_source': 'model',
        'weight_mode': 'brutal',
        'overround': 1.10,
        'slippage_prob': 0.020,
        'edge_threshold': 0.08,
        'kelly_frac': 0.05,
        'max_stake': 0.01,
    },
    {
        'label': 'DUMB DC-only baseline',
        'prob_source': 'dc',
        'weight_mode': 'base',
        'overround': 1.07,
        'slippage_prob': 0.010,
        'edge_threshold': 0.05,
        'kelly_frac': 0.10,
        'max_stake': 0.02,
    },
    {
        'label': 'DUMB closing-only baseline',
        'prob_source': 'closing',
        'weight_mode': 'base',
        'overround': 1.07,
        'slippage_prob': 0.010,
        'edge_threshold': 0.05,
        'kelly_frac': 0.10,
        'max_stake': 0.02,
    },
    {
        'label': 'DUMB random baseline',
        'prob_source': 'random',
        'weight_mode': 'base',
        'overround': 1.07,
        'slippage_prob': 0.010,
        'edge_threshold': 0.05,
        'kelly_frac': 0.10,
        'max_stake': 0.02,
    },
]

results = {}

for cfg in configs:
    bdf = run_strategy(
        bt,
        label=cfg['label'],
        prob_source=cfg['prob_source'],
        weight_mode=cfg['weight_mode'],
        overround=cfg['overround'],
        slippage_prob=cfg['slippage_prob'],
        edge_threshold=cfg['edge_threshold'],
        kelly_frac=cfg['kelly_frac'],
        max_stake=cfg['max_stake'],
        ban_same_checkpoint_after_state_change=True,
        verbose=True
    )

    results[cfg['label']] = bdf

print()
print("Checkpoint breakdown: MODEL base conservative")
base_bdf = results['MODEL base conservative']
print(summarize_by_checkpoint(base_bdf).to_string())

print()
print("Side breakdown: MODEL base conservative")
print(summarize_by_side(base_bdf).to_string())

print()
print("Most common score states: MODEL base conservative")
print(summarize_by_score_state(base_bdf).head(20).to_string())

print()
print("Sensitivity grid for MODEL strategy")
grid_rows = []

for weight_mode in ['loose', 'base', 'brutal']:
    for overround in [1.05, 1.07, 1.10]:
        for slippage_prob in [0.005, 0.010, 0.020]:
            for edge_threshold in [0.03, 0.05, 0.08]:
                bdf = run_strategy(
                    bt,
                    label='silent',
                    prob_source='model',
                    weight_mode=weight_mode,
                    overround=overround,
                    slippage_prob=slippage_prob,
                    edge_threshold=edge_threshold,
                    kelly_frac=0.10,
                    max_stake=0.02,
                    ban_same_checkpoint_after_state_change=True,
                    verbose=False
                )

                if bdf is None or len(bdf) == 0:
                    grid_rows.append({
                        'weight_mode': weight_mode,
                        'overround': overround,
                        'slippage_prob': slippage_prob,
                        'edge_threshold': edge_threshold,
                        'n_bets': 0,
                        'hit_rate': np.nan,
                        'stake': 0.0,
                        'pnl': 0.0,
                        'roi_pct': np.nan,
                        'avg_edge': np.nan,
                        'avg_w': np.nan
                    })
                else:
                    stake = bdf['stake'].sum()
                    pnl = bdf['pnl'].sum()

                    grid_rows.append({
                        'weight_mode': weight_mode,
                        'overround': overround,
                        'slippage_prob': slippage_prob,
                        'edge_threshold': edge_threshold,
                        'n_bets': len(bdf),
                        'hit_rate': bdf['won'].mean(),
                        'stake': stake,
                        'pnl': pnl,
                        'roi_pct': pnl / stake * 100 if stake > 0 else np.nan,
                        'avg_edge': bdf['edge'].mean(),
                        'avg_w': bdf['w_used'].mean()
                    })

grid = pd.DataFrame(grid_rows)
grid_sorted = grid.sort_values(['weight_mode', 'overround', 'slippage_prob', 'edge_threshold'])

print(
    grid_sorted[
        [
            'weight_mode',
            'overround',
            'slippage_prob',
            'edge_threshold',
            'n_bets',
            'hit_rate',
            'roi_pct',
            'pnl',
            'stake',
            'avg_edge',
            'avg_w'
        ]
    ].round(4).to_string(index=False)
)

bt.to_parquet(PROCESSED / 'backtest_results_with_dc_and_state_flags.parquet', index=False)

if base_bdf is not None and len(base_bdf) > 0:
    base_bdf.to_parquet(PROCESSED / 'bets_model_base_conservative.parquet', index=False)
    base_bdf.to_csv(PROCESSED / 'bets_model_base_conservative.csv', index=False)

grid_sorted.to_csv(PROCESSED / 'synthetic_odds_stress_grid.csv', index=False)

print()
print("Saved:")
print("  backtest_results_with_dc_and_state_flags.parquet")
print("  bets_model_base_conservative.parquet / .csv")
print("  synthetic_odds_stress_grid.csv")

In [ ]:
import numpy as np

def bootstrap_roi(bdf, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    rois = []
    n = len(bdf)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        sample = bdf.iloc[idx]
        s = sample['stake'].sum()
        p = sample['pnl'].sum()
        if s > 0:
            rois.append(p/s * 100)
    rois = np.array(rois)
    return np.mean(rois), np.percentile(rois, 2.5), np.percentile(rois, 97.5)

import pandas as pd
from pathlib import Path
PROCESSED = Path('/content/drive/MyDrive/soccer-betting/data/processed/api_football')
base_bdf = pd.read_parquet(PROCESSED / 'bets_model_base_conservative.parquet')

mean, lo, hi = bootstrap_roi(base_bdf)
print(f"MODEL base conservative: ROI = {mean:.2f}% [95% CI: {lo:.2f}% to {hi:.2f}%]  n={len(base_bdf)}")

# Single-match mechanism example


In [ ]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from scipy.stats import poisson

PROCESSED = Path('/content/drive/MyDrive/soccer-betting/data/processed/api_football')
FID = 1379231
BANKROLL_START = 10_000

OVERROUND = 1.07
SLIPPAGE = 0.010
EDGE_THRESHOLD = 0.05
KELLY_FRAC = 0.10
MAX_STAKE_FRAC = 0.02

with open(PROCESSED / 'hazard_models.pkl', 'rb') as f:
    haz = pickle.load(f)
hazard_h, hazard_a = haz['hazard_h'], haz['hazard_a']
HAZARD_FEATS = haz['features']
F_IDX = {f: i for i, f in enumerate(HAZARD_FEATS)}
beta_h = hazard_h.coef_.astype(np.float32)
beta_a = hazard_a.coef_.astype(np.float32)
b0_h, b0_a = float(hazard_h.intercept_), float(hazard_a.intercept_)

events = pd.read_parquet(PROCESSED / 'events_clean.parquet')
fixtures = pd.read_parquet(PROCESSED / 'fixtures_usable.parquet')
mf = pd.read_parquet(PROCESSED / 'match_features_v2.parquet')
cal_df = pd.read_parquet(PROCESSED / 'sim_calibration.parquet')

FD_DIR = Path('/content/drive/MyDrive/soccer-betting/data/processed/fd')
fd = pd.read_parquet(FD_DIR / 'matches.parquet')
fd['date'] = pd.to_datetime(fd['date'], utc=True, errors='coerce')

fx = fixtures[fixtures['fixture_id'] == FID].iloc[0]
mf_row = mf[mf['fixture_id'] == FID].iloc[0]
cal_row = cal_df[cal_df['fixture_id'] == FID].iloc[0]
home_id = fx['home_team_id']
dc_lh, dc_la = mf_row['dc_lambda_home'], mf_row['dc_lambda_away']
alpha_h, alpha_a = cal_row['alpha_h'], cal_row['alpha_a']

fd_match = fd[(fd['date'].dt.date == fx['date'].date())
              & fd['home_team_name'].str.contains('Chelsea', case=False, na=False)
              & fd['away_team_name'].str.contains('Burnley', case=False, na=False)]
if len(fd_match) > 0:
    fdr = fd_match.iloc[0]
    closing_p = (fdr['B365C_p_home'], fdr['B365C_p_draw'], fdr['B365C_p_away'])
    print(f"B365 closing odds: H={fdr['B365CH']:.2f} D={fdr['B365CD']:.2f} A={fdr['B365CA']:.2f}")
    print(f"Implied (no overround): P(H)={closing_p[0]:.3f} P(D)={closing_p[1]:.3f} P(A)={closing_p[2]:.3f}")
else:
    closing_p = (mf_row['dc_p_home'], mf_row['dc_p_draw'], mf_row['dc_p_away'])
    print(f"No FD match found — using DC as closing anchor")

print(f"\nMatch: {fx['home_team_name']} {int(fx['home_score'])}-{int(fx['away_score'])} {fx['away_team_name']}")
print(f"Date: {fx['date']}, Competition: {fx['target_slug']}")
print(f"DC pre-match: lh={dc_lh:.2f}, la={dc_la:.2f}, "
      f"P(home)={mf_row['dc_p_home']:.3f} P(draw)={mf_row['dc_p_draw']:.3f} P(away)={mf_row['dc_p_away']:.3f}")
print(f"Calibrated alphas: alpha_h={alpha_h:.3f}, alpha_a={alpha_a:.3f}")
print(f"Bankroll start: ${BANKROLL_START:,}")
print()

m_events = events[events['fixture_id'] == FID].sort_values(['minute', 'extra']).copy()
m_events['side'] = m_events['team_id'].apply(lambda t: 'H' if t == home_id else 'A')

print("Event timeline:")
for _, e in m_events.iterrows():
    if e['type'] in ('Goal',) or (e['type'] == 'Card' and 'Red' in str(e['detail'])):
        team = fx['home_team_name'] if e['side'] == 'H' else fx['away_team_name']
        marker = '⚽' if e['type'] == 'Goal' else '🟥'
        print(f"  min {int(e['minute']):>3}  {marker} {team:<12s}  {e.get('player_name') or '?'}  ({e.get('detail') or ''})")
print()

def simulate(start_minute, h0, a0, hr0, ar0, n_sims=4000, seed=42):
    rng = np.random.default_rng(seed)
    log_lh = np.log(max(dc_lh, 0.05)); log_la = np.log(max(dc_la, 0.05))
    h_score = np.full(n_sims, h0, dtype=np.int32)
    a_score = np.full(n_sims, a0, dtype=np.int32)
    h_man_up = float(ar0 > hr0); a_man_up = float(hr0 > ar0)

    for minute in range(start_minute, 90):
        tf = np.zeros(len(HAZARD_FEATS), dtype=np.float32)
        if minute < 15:   tf[F_IDX['min_0_15']]  = 1
        elif minute < 30: tf[F_IDX['min_15_30']] = 1
        elif minute < 45: tf[F_IDX['min_30_45']] = 1
        elif minute < 60: pass
        elif minute < 75: tf[F_IDX['min_60_75']] = 1
        else:             tf[F_IDX['min_75_90']] = 1
        tf[F_IDX['h_man_up']] = h_man_up
        tf[F_IDX['a_man_up']] = a_man_up
        tf[F_IDX['log_dc_lh']] = log_lh
        tf[F_IDX['log_dc_la']] = log_la

        base_h = b0_h + np.dot(beta_h, tf)
        base_a = b0_a + np.dot(beta_a, tf)
        sd = h_score - a_score
        hl = (sd > 0).astype(np.float32); al = (sd < 0).astype(np.float32); bg = (np.abs(sd) >= 2).astype(np.float32)
        dh = beta_h[F_IDX['h_leading']]*hl + beta_h[F_IDX['a_leading']]*al + beta_h[F_IDX['big_lead']]*bg
        da = beta_a[F_IDX['h_leading']]*hl + beta_a[F_IDX['a_leading']]*al + beta_a[F_IDX['big_lead']]*bg

        lam_h = np.exp(base_h + dh) * alpha_h
        lam_a = np.exp(base_a + da) * alpha_a
        h_score += rng.poisson(lam_h).astype(np.int32)
        a_score += rng.poisson(lam_a).astype(np.int32)

    return (float((h_score > a_score).mean()),
            float((h_score == a_score).mean()),
            float((h_score < a_score).mean()))

def dc_state_probs(h_sc, a_sc, minutes_left, lh, la, max_extra=10):
    frac = max(minutes_left, 0) / 90.0
    lam_h = max(lh * frac, 0); lam_a = max(la * frac, 0)
    if lam_h < 1e-9 and lam_a < 1e-9:
        if h_sc > a_sc: return 1.0, 0.0, 0.0
        if h_sc < a_sc: return 0.0, 0.0, 1.0
        return 0.0, 1.0, 0.0
    pe_h = poisson.pmf(np.arange(max_extra+1), lam_h)
    pe_a = poisson.pmf(np.arange(max_extra+1), lam_a)
    M = np.outer(pe_h, pe_a)
    fh = h_sc + np.arange(max_extra+1).reshape(-1,1)
    fa = a_sc + np.arange(max_extra+1).reshape(1,-1)
    p_home = M[fh > fa].sum(); p_draw = M[fh == fa].sum(); p_away = M[fh < fa].sum()
    Z = p_home + p_draw + p_away
    return p_home/Z, p_draw/Z, p_away/Z

def synth_market_w(t, h_sc, a_sc, h_rd, a_rd, state_changed):
    w0 = 0.65
    w = w0 * np.exp(-0.015 * t) * np.exp(-0.55 * abs(h_sc - a_sc))
    if h_sc + a_sc > 0: w = min(w, 0.35)
    if state_changed: w = min(w, 0.25)
    if (h_rd + a_rd) > 0: w = min(w, 0.20)
    if t >= 70: w = min(w, 0.15)
    if t >= 80: w = min(w, 0.10)
    return float(np.clip(w, 0.02, 0.70))

CHECKPOINTS = list(range(0, 90, 5))
bankroll = BANKROLL_START
bets_log = []
prev_state = None

print(f"{'min':>4s} {'score':>5s} {'state':10s}  {'side':<5s} "
      f"{'p_mod':>6s} {'p_mkt':>6s} {'edge':>7s} {'odds':>6s} "
      f"{'stake$':>9s} {'bank$':>11s}  {'result':<20s}")
print('-' * 115)

for cp in CHECKPOINTS:
    h_sc = a_sc = h_rd = a_rd = 0
    for _, e in m_events.iterrows():
        if e['minute'] >= cp:
            break
        if e['side'] == 'H':
            if e['type'] == 'Goal': h_sc += 1
            elif e['type'] == 'Card' and 'Red' in str(e['detail']): h_rd += 1
        elif e['side'] == 'A':
            if e['type'] == 'Goal': a_sc += 1
            elif e['type'] == 'Card' and 'Red' in str(e['detail']): a_rd += 1

    cur_state = (h_sc, a_sc, h_rd, a_rd)
    state_changed = (prev_state is not None and cur_state != prev_state)
    if state_changed:
        prev_state = cur_state
        continue
    prev_state = cur_state

    p_h, p_d, p_a = simulate(cp, h_sc, a_sc, h_rd, a_rd, n_sims=4000)
    dc_h, dc_d, dc_a = dc_state_probs(h_sc, a_sc, 90-cp, dc_lh, dc_la)
    w = synth_market_w(cp, h_sc, a_sc, h_rd, a_rd, state_changed)
    mp_h = w * closing_p[0] + (1-w) * dc_h
    mp_d = w * closing_p[1] + (1-w) * dc_d
    mp_a = w * closing_p[2] + (1-w) * dc_a
    Z = mp_h + mp_d + mp_a
    mp_h, mp_d, mp_a = mp_h/Z, mp_d/Z, mp_a/Z

    actual = 'H' if int(fx['home_score']) > int(fx['away_score']) else ('A' if int(fx['home_score']) < int(fx['away_score']) else 'D')

    state_str = ""
    if h_rd > 0: state_str += f"H-{h_rd}r "
    if a_rd > 0: state_str += f"A-{a_rd}r "

    fired_any = False
    for side_name, pm, mp, letter in [('Chel', p_h, mp_h, 'H'),
                                       ('Draw', p_d, mp_d, 'D'),
                                       ('Burn', p_a, mp_a, 'A')]:
        mp_after = mp * OVERROUND + SLIPPAGE
        if mp_after >= 0.999: continue
        edge = pm - mp_after
        if edge < EDGE_THRESHOLD: continue
        odds = 1.0 / mp_after
        b = odds - 1; q = 1 - pm
        if b * pm <= q: continue
        kelly = (b * pm - q) / b
        stake_frac = min(KELLY_FRAC * kelly, MAX_STAKE_FRAC)
        if stake_frac <= 0: continue

        stake_dollars = stake_frac * bankroll
        won = (letter == actual)
        pnl_dollars = stake_dollars * (odds - 1) if won else -stake_dollars
        bankroll += pnl_dollars

        result_str = f"✓ WIN +${pnl_dollars:,.0f}" if won else f"✗ LOSS -${-pnl_dollars:,.0f}"
        print(f"{cp:>4d} {h_sc}-{a_sc:<3d} {state_str:10s}  {side_name:<5s} "
              f"{pm:.3f} {mp_after:.3f} {edge*100:>+5.1f}% {odds:>5.2f} "
              f"${stake_dollars:>8.0f} ${bankroll:>10,.0f}  {result_str}")
        bets_log.append({
            'min': cp, 'score': f"{h_sc}-{a_sc}", 'side': side_name,
            'p_model': pm, 'p_market': mp_after, 'edge': edge,
            'odds': odds, 'stake_$': stake_dollars, 'won': won,
            'pnl_$': pnl_dollars, 'bankroll_after': bankroll,
        })
        fired_any = True

    if not fired_any:
        print(f"{cp:>4d} {h_sc}-{a_sc:<3d} {state_str:10s}  {'--':<5s} "
              f"{'':>6s} {'':>6s} {'':>7s} {'':>6s} "
              f"{'':>9s} ${bankroll:>10,.0f}  "
              f"(p={p_h:.2f}/{p_d:.2f}/{p_a:.2f} mkt={mp_h*OVERROUND+SLIPPAGE:.2f}/{mp_d*OVERROUND+SLIPPAGE:.2f}/{mp_a*OVERROUND+SLIPPAGE:.2f})")

print('-' * 115)
print(f"\nFinal bankroll: ${bankroll:,.2f}")
print(f"Net PnL:        ${bankroll - BANKROLL_START:+,.2f}")
print(f"ROI on starting bankroll: {(bankroll - BANKROLL_START)/BANKROLL_START*100:+.2f}%")
print(f"Total bets placed: {len(bets_log)}")
if bets_log:
    bdf = pd.DataFrame(bets_log)
    print(f"Hit rate: {bdf['won'].mean()*100:.1f}%  ({int(bdf['won'].sum())}/{len(bdf)})")
    print(f"Total staked: ${bdf['stake_$'].sum():,.0f}")
    print(f"Avg edge:     {bdf['edge'].mean()*100:.1f}%")
    print(f"\nDetailed bet log:")
    print(bdf[['min', 'score', 'side', 'p_model', 'p_market', 'edge',
               'odds', 'stake_$', 'won', 'pnl_$', 'bankroll_after']].to_string(index=False))

# Final paper figures


In [ ]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

PROCESSED = Path('/content/drive/MyDrive/soccer-betting/data/processed/api_football')
FIG_DIR = Path('/content/drive/MyDrive/soccer-betting/paper/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 130,
    'savefig.dpi': 200,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

def save(fig, name):
    fig.savefig(FIG_DIR / f'{name}.pdf', bbox_inches='tight')
    fig.savefig(FIG_DIR / f'{name}.png', bbox_inches='tight')
    print(f"  saved -> {name}.pdf + .png")

print("FIG 1: trigger comparison")

triggers = [
    ('All matches', 7538, 54.9, (54, 56), 'baseline'),
    ('Any goal\nbefore 35\'', 4510, 67.6, (66, 69), 'baseline'),
    ('Big favorite\nplaying', 2396, 52.7, (51, 55), 'baseline'),
    ('Favorite\nconceded first', 2610, 69.2, (67, 71), 'trigger'),
    ('Favorite conceded\nfirst before 35\'', 1683, 78.6, (77, 80), 'trigger'),
    ('Big favorite conceded\nfirst before 35\'', 395, 88.1, (85, 91), 'strict'),
]

fig, ax = plt.subplots(figsize=(9, 4.5))
labels = [t[0] for t in triggers]
rates = [t[2] for t in triggers]
ci_lo = [t[3][0] for t in triggers]
ci_hi = [t[3][1] for t in triggers]
ns = [t[1] for t in triggers]
colors = ['#a8a8a8' if t[4] == 'baseline' else '#3b6fb8' if t[4] == 'trigger' else '#1f4e8c' for t in triggers]

x = np.arange(len(triggers))
ax.bar(x, rates, color=colors, edgecolor='black', linewidth=0.5, width=0.7)
ax.errorbar(x, rates, yerr=[np.array(rates)-np.array(ci_lo), np.array(ci_hi)-np.array(rates)],
            fmt='none', color='black', capsize=4, lw=1)

for i, (r, n) in enumerate(zip(rates, ns)):
    ax.text(i, r + 4, f'{r:.1f}%\nn={n:,}', ha='center', fontsize=8.5)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8.5)
ax.set_ylabel('BTTS rate (%)')
ax.set_ylim(0, 100)
ax.set_title('BTTS conditional outcome rates by trigger condition (Wilson 95% CI)')
ax.axhline(54.9, color='red', ls='--', lw=0.8, alpha=0.5)
ax.text(5.4, 56, 'baseline', color='red', fontsize=8, ha='right', alpha=0.7)

legend_elements = [
    mpatches.Patch(color='#a8a8a8', label='Baseline conditions'),
    mpatches.Patch(color='#3b6fb8', label='State-space triggers'),
    mpatches.Patch(color='#1f4e8c', label='Strictest trigger'),
]
ax.legend(handles=legend_elements, loc='upper left', frameon=False)
plt.tight_layout()
save(fig, 'fig1_trigger_btts_rates')
plt.close()

print("FIG 2: equity curve")

bets = pd.read_parquet(PROCESSED / 'bets_model_base_conservative.parquet')
bets = bets.sort_values(['fixture_id', 'checkpoint']).reset_index(drop=True)
bets['cum_pnl'] = bets['pnl'].cumsum()
bets['cum_stake'] = bets['stake'].cumsum()
bets['bet_num'] = range(1, len(bets) + 1)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(bets['bet_num'], bets['cum_pnl'], lw=1.4, color='#1f4e8c', label='Cumulative PnL')
ax.fill_between(bets['bet_num'], 0, bets['cum_pnl'],
                where=bets['cum_pnl'] >= 0, alpha=0.15, color='#3b6fb8')
ax.fill_between(bets['bet_num'], 0, bets['cum_pnl'],
                where=bets['cum_pnl'] < 0, alpha=0.15, color='#c44')
ax.axhline(0, color='black', lw=0.6, alpha=0.5)

final_pnl = bets['cum_pnl'].iloc[-1]
final_stake = bets['cum_stake'].iloc[-1]
roi = final_pnl / final_stake * 100

ax.set_xlabel('Bet number')
ax.set_ylabel('Cumulative PnL (units)')
ax.set_title(f'Equity curve, MODEL base conservative (n={len(bets)} bets, total ROI {roi:.2f}%)')
ax.legend(loc='upper left', frameon=False)
ax.grid(True, alpha=0.3, lw=0.4)
plt.tight_layout()
save(fig, 'fig2_equity_curve')
plt.close()

print("FIG 3: Chelsea-Burnley trajectory")

FID = 1379231

with open(PROCESSED / 'hazard_models.pkl', 'rb') as f:
    haz = pickle.load(f)
hazard_h, hazard_a = haz['hazard_h'], haz['hazard_a']
HAZARD_FEATS = haz['features']
F_IDX = {f: i for i, f in enumerate(HAZARD_FEATS)}
beta_h = hazard_h.coef_.astype(np.float32)
beta_a = hazard_a.coef_.astype(np.float32)
b0_h, b0_a = float(hazard_h.intercept_), float(hazard_a.intercept_)

events = pd.read_parquet(PROCESSED / 'events_clean.parquet')
fixtures = pd.read_parquet(PROCESSED / 'fixtures_usable.parquet')
mf = pd.read_parquet(PROCESSED / 'match_features_v2.parquet')
cal_df = pd.read_parquet(PROCESSED / 'sim_calibration.parquet')

fx = fixtures[fixtures['fixture_id'] == FID].iloc[0]
mf_row = mf[mf['fixture_id'] == FID].iloc[0]
cal_row = cal_df[cal_df['fixture_id'] == FID].iloc[0]
home_id = fx['home_team_id']
dc_lh, dc_la = mf_row['dc_lambda_home'], mf_row['dc_lambda_away']
alpha_h, alpha_a = cal_row['alpha_h'], cal_row['alpha_a']

m_events = events[events['fixture_id'] == FID].sort_values(['minute', 'extra']).copy()
m_events['side'] = m_events['team_id'].apply(lambda t: 'H' if t == home_id else 'A')

def simulate(start_minute, h0, a0, hr0, ar0, n_sims=4000, seed=42):
    rng = np.random.default_rng(seed)
    log_lh = np.log(max(dc_lh, 0.05)); log_la = np.log(max(dc_la, 0.05))
    h_score = np.full(n_sims, h0, dtype=np.int32)
    a_score = np.full(n_sims, a0, dtype=np.int32)
    h_man_up = float(ar0 > hr0); a_man_up = float(hr0 > ar0)

    for minute in range(start_minute, 90):
        tf = np.zeros(len(HAZARD_FEATS), dtype=np.float32)
        if minute < 15:   tf[F_IDX['min_0_15']]  = 1
        elif minute < 30: tf[F_IDX['min_15_30']] = 1
        elif minute < 45: tf[F_IDX['min_30_45']] = 1
        elif minute < 60: pass
        elif minute < 75: tf[F_IDX['min_60_75']] = 1
        else:             tf[F_IDX['min_75_90']] = 1
        tf[F_IDX['h_man_up']] = h_man_up
        tf[F_IDX['a_man_up']] = a_man_up
        tf[F_IDX['log_dc_lh']] = log_lh
        tf[F_IDX['log_dc_la']] = log_la

        base_h = b0_h + np.dot(beta_h, tf)
        base_a = b0_a + np.dot(beta_a, tf)
        sd = h_score - a_score
        hl = (sd > 0).astype(np.float32); al = (sd < 0).astype(np.float32); bg = (np.abs(sd) >= 2).astype(np.float32)
        dh = beta_h[F_IDX['h_leading']]*hl + beta_h[F_IDX['a_leading']]*al + beta_h[F_IDX['big_lead']]*bg
        da = beta_a[F_IDX['h_leading']]*hl + beta_a[F_IDX['a_leading']]*al + beta_a[F_IDX['big_lead']]*bg

        lam_h = np.exp(base_h + dh) * alpha_h
        lam_a = np.exp(base_a + da) * alpha_a
        h_score += rng.poisson(lam_h).astype(np.int32)
        a_score += rng.poisson(lam_a).astype(np.int32)

    return (float((h_score > a_score).mean()),
            float((h_score == a_score).mean()),
            float((h_score < a_score).mean()))

trace_minutes = list(range(0, 91))
probs = {'home': [], 'draw': [], 'away': []}
for m in trace_minutes:
    h_sc = a_sc = h_rd = a_rd = 0
    for _, e in m_events.iterrows():
        if e['minute'] >= m:
            break
        if e['side'] == 'H':
            if e['type'] == 'Goal': h_sc += 1
            elif e['type'] == 'Card' and 'Red' in str(e['detail']): h_rd += 1
        elif e['side'] == 'A':
            if e['type'] == 'Goal': a_sc += 1
            elif e['type'] == 'Card' and 'Red' in str(e['detail']): a_rd += 1
    p_h, p_d, p_a = simulate(m, h_sc, a_sc, h_rd, a_rd, n_sims=2000, seed=m)
    probs['home'].append(p_h)
    probs['draw'].append(p_d)
    probs['away'].append(p_a)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trace_minutes, probs['home'], color='#1f4e8c', lw=2, label='P(Chelsea win)')
ax.plot(trace_minutes, probs['draw'], color='#666', lw=2, label='P(Draw)')
ax.plot(trace_minutes, probs['away'], color='#c44', lw=2, label='P(Burnley win)')

ax.axvline(4, color='#1f4e8c', ls='--', lw=0.8, alpha=0.7)
ax.text(4, 1.02, 'Chelsea goal\nmin 4', ha='center', fontsize=8, color='#1f4e8c')
ax.axvline(72, color='red', ls='--', lw=0.8, alpha=0.7)
ax.text(72, 1.02, 'Fofana red\nmin 72', ha='center', fontsize=8, color='red')
ax.axvline(90, color='#c44', ls='--', lw=0.8, alpha=0.7)
ax.text(90, 1.02, 'Burnley goal\nmin 90', ha='center', fontsize=8, color='#c44')

ax.axvline(80, color='gold', ls=':', lw=2, alpha=0.8)
ax.annotate('Bet fires:\n$70 on Draw\n@ 12.24 odds',
            xy=(80, 0.14), xytext=(60, 0.42),
            fontsize=9, color='#a06800',
            arrowprops=dict(arrowstyle='->', color='#a06800', lw=1))

ax.set_xlabel('Match minute')
ax.set_ylabel('Probability')
ax.set_title(f'Probability trajectory: Chelsea {int(fx["home_score"])}-{int(fx["away_score"])} Burnley (fid={FID})')
ax.set_xlim(-2, 95)
ax.set_ylim(-0.02, 1.08)
ax.legend(loc='center right', frameon=False)
ax.grid(True, alpha=0.3, lw=0.4)
plt.tight_layout()
save(fig, 'fig3_chelsea_burnley_trajectory')
plt.close()

print("FIG 4: pre-match calibration scatter")

bt = pd.read_parquet(PROCESSED / 'backtest_results.parquet')
pm = bt[bt['checkpoint'] == 0].copy()

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, side, color in zip(axes, ['home', 'draw', 'away'], ['#1f4e8c', '#666', '#c44']):
    ax.scatter(pm[f'market_p_{side}'], pm[f'model_p_{side}'],
               s=14, alpha=0.4, color=color)
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.5)
    ax.set_xlabel(f'Market P({side})')
    ax.set_ylabel(f'Model P({side})')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.set_title(f'{side.capitalize()}', color=color)

fig.suptitle(f'Pre-match calibration: model vs B365 closing (n={len(pm)})', fontsize=11)
plt.tight_layout()
save(fig, 'fig4_calibration_scatter')
plt.close()

print("FIG 5: sensitivity grid")

grid = pd.read_csv(PROCESSED / 'synthetic_odds_stress_grid.csv')

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, mode in zip(axes, ['loose', 'base', 'brutal']):
    sub = grid[grid['weight_mode'] == mode].copy()
    pivot = sub.pivot_table(index='overround', columns='edge_threshold',
                             values='roi_pct', aggfunc='mean')
    im = ax.imshow(pivot.values, cmap='RdBu_r', vmin=-30, vmax=100, aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{c:.2f}' for c in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{i:.2f}' for i in pivot.index])
    ax.set_xlabel('Edge threshold')
    ax.set_title(f"weight = '{mode}'")
    if mode == 'loose':
        ax.set_ylabel('Overround')
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            color = 'white' if abs(val) > 50 else 'black'
            ax.text(j, i, f'{val:.0f}%', ha='center', va='center',
                    fontsize=9, color=color, fontweight='bold')

cbar = fig.colorbar(im, ax=axes, shrink=0.7, pad=0.02)
cbar.set_label('ROI (%)')
fig.suptitle('Synthetic-odds backtest ROI across weight schedules, overrounds, edge thresholds',
              fontsize=11)
save(fig, 'fig5_sensitivity_grid')
plt.close()

print("FIG 6: hazard model coefficients")

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(HAZARD_FEATS))
width = 0.4
ax.barh(x - width/2, beta_h, width, color='#1f4e8c', label='Home goal hazard')
ax.barh(x + width/2, beta_a, width, color='#c44', label='Away goal hazard')
ax.set_yticks(x)
ax.set_yticklabels(HAZARD_FEATS)
ax.axvline(0, color='black', lw=0.6)
ax.set_xlabel('Log-rate coefficient')
ax.set_title('Hazard model coefficients (Poisson regression)')
ax.legend(loc='lower right', frameon=False)
ax.grid(True, axis='x', alpha=0.3, lw=0.4)
plt.tight_layout()
save(fig, 'fig6_hazard_coefficients')
plt.close()

print(f"\nAll figures saved to: {FIG_DIR}")
for f in sorted(FIG_DIR.glob('*.pdf')):
    print(f"  {f.name}")

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches

FIG_DIR = Path('/content/drive/MyDrive/soccer-betting/paper/figures')
PROCESSED = Path('/content/drive/MyDrive/soccer-betting/data/processed/api_football')

plt.rcParams.update({
    'font.size': 10, 'axes.labelsize': 11, 'axes.titlesize': 12,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9,
    'figure.dpi': 130, 'savefig.dpi': 200,
    'axes.spines.top': False, 'axes.spines.right': False,
})

def save(fig, name):
    fig.savefig(FIG_DIR / f'{name}.pdf', bbox_inches='tight')
    fig.savefig(FIG_DIR / f'{name}.png', bbox_inches='tight')
    print(f"  saved -> {name}")

print("Displaying existing 6 figures:\n")
for fp in sorted(FIG_DIR.glob('fig[1-6]*.png')):
    print(f"  {fp.name}")
    display(Image(filename=str(fp)))

print("\nFIG 7: model vs dumb baselines\n")

strategies = pd.DataFrame([
    {'label': 'MODEL\nloose',  'n': 1497, 'roi': -1.66,  'family': 'model'},
    {'label': 'MODEL\nbase',   'n': 532,  'roi': 10.51,  'family': 'model'},
    {'label': 'MODEL\nbrutal', 'n': 161,  'roi': 88.71,  'family': 'model'},
    {'label': 'DUMB\nDC-only', 'n': 197,  'roi': -2.53,  'family': 'baseline'},
    {'label': 'DUMB\nclosing', 'n': 3715, 'roi': -9.69,  'family': 'baseline'},
    {'label': 'DUMB\nrandom',  'n': 5189, 'roi': -11.07, 'family': 'baseline'},
])

fig, ax = plt.subplots(figsize=(9, 4.8))
colors = ['#1f4e8c' if f == 'model' else '#a8a8a8' for f in strategies['family']]
ax.bar(strategies['label'], strategies['roi'], color=colors,
       edgecolor='black', linewidth=0.5, width=0.7)
ax.axhline(0, color='black', lw=0.8)

for i, (roi, n) in enumerate(zip(strategies['roi'], strategies['n'])):
    offset = 4 if roi >= 0 else -7
    ax.text(i, roi + offset, f'{roi:+.1f}%\nn={n:,}',
            ha='center', fontsize=8.5,
            color='black' if roi >= 0 else 'darkred')

ax.set_ylabel('ROI (%)')
ax.set_title('Model strategies vs dumb baselines (identical friction parameters)')
ax.set_ylim(min(strategies['roi']) - 18, max(strategies['roi']) + 18)
ax.legend(handles=[mpatches.Patch(color='#1f4e8c', label='Hazard model strategies'),
                   mpatches.Patch(color='#a8a8a8', label='Dumb baselines')],
          loc='upper right', frameon=False)
plt.tight_layout()
save(fig, 'fig7_model_vs_baselines')
display(fig)
plt.close()

print("\nFIG 8: ROI + bet count by checkpoint\n")

cp_data = pd.DataFrame([
    (0, 51, -15.17), (5, 42, -29.54), (10, 36, -17.31), (15, 31, -38.48),
    (20, 30, 7.54), (25, 24, 29.05), (30, 28, 34.36), (35, 24, 33.12),
    (40, 20, 63.18), (45, 19, 89.96), (50, 15, 86.78), (55, 19, 65.01),
    (60, 23, -29.66), (65, 30, -29.99), (70, 38, -15.71), (75, 49, 9.28),
    (80, 45, 72.53), (85, 8, 12.19),
], columns=['cp', 'n', 'roi'])

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

bar_colors = ['#3b6fb8' if r >= 0 else '#c44' for r in cp_data['roi']]
ax1.bar(cp_data['cp'], cp_data['roi'], width=3.5, color=bar_colors,
        edgecolor='black', linewidth=0.4, alpha=0.85)
ax1.axhline(0, color='black', lw=0.8)
ax1.set_xlabel('Match minute (checkpoint)')
ax1.set_ylabel('ROI (%)', color='#1f4e8c')
ax1.tick_params(axis='y', labelcolor='#1f4e8c')
ax1.set_xticks(cp_data['cp'])

ax2.plot(cp_data['cp'], cp_data['n'], color='black', marker='o',
         lw=1.5, markersize=5, label='Number of bets')
ax2.set_ylabel('Number of bets fired', color='black')
ax2.set_ylim(0, max(cp_data['n']) + 10)
ax2.spines['top'].set_visible(False)
ax2.legend(loc='upper right', frameon=False)

ax1.set_title('ROI and bet count by checkpoint (MODEL base conservative)')
plt.tight_layout()
save(fig, 'fig8_roi_by_checkpoint')
display(fig)
plt.close()

print("\nFIG 9: bet firing heatmap by score state x minute bucket\n")

bets = pd.read_parquet(PROCESSED / 'bets_model_base_conservative.parquet')
bets['score_state'] = bets['score_home'].astype(str) + '-' + bets['score_away'].astype(str)

def cp_to_bucket(cp):
    if cp < 15:  return '0-15'
    if cp < 30:  return '15-30'
    if cp < 45:  return '30-45'
    if cp < 60:  return '45-60'
    if cp < 75:  return '60-75'
    return '75-90'

bets['bucket'] = bets['checkpoint'].apply(cp_to_bucket)

common_states = bets['score_state'].value_counts().head(8).index.tolist()
common_states_sorted = sorted(common_states, key=lambda s: (sum(int(x) for x in s.split('-')), s))

heatmap_n = bets[bets['score_state'].isin(common_states_sorted)].pivot_table(
    index='score_state', columns='bucket', values='stake', aggfunc='count', fill_value=0
)
bucket_order = ['0-15', '15-30', '30-45', '45-60', '60-75', '75-90']
heatmap_n = heatmap_n.reindex(index=common_states_sorted, columns=bucket_order, fill_value=0)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(heatmap_n.values, cmap='Blues', aspect='auto')

for i in range(len(heatmap_n.index)):
    for j in range(len(heatmap_n.columns)):
        v = int(heatmap_n.values[i, j])
        if v > 0:
            color = 'white' if v > heatmap_n.values.max() * 0.5 else 'black'
            ax.text(j, i, str(v), ha='center', va='center',
                    fontsize=10, color=color, fontweight='bold')

ax.set_xticks(range(len(heatmap_n.columns)))
ax.set_xticklabels(heatmap_n.columns)
ax.set_yticks(range(len(heatmap_n.index)))
ax.set_yticklabels(heatmap_n.index)
ax.set_xlabel('Match minute bucket')
ax.set_ylabel('Score state (home-away)')
ax.set_title('Where the strategy fires: bet count by (score state, time bucket)')
cbar = fig.colorbar(im, ax=ax, shrink=0.7)
cbar.set_label('Bets fired')
plt.tight_layout()
save(fig, 'fig9_firing_heatmap')
display(fig)
plt.close()

print("\n" + "=" * 60)
print(f"ALL FIGURES IN: {FIG_DIR}")
print("=" * 60)
for fp in sorted(FIG_DIR.glob('*.pdf')):
    print(f"  {fp.name}")

# Pre-match scoring diagnostics


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED = Path('/content/drive/MyDrive/soccer-betting/data/processed/api_football')
bt = pd.read_parquet(PROCESSED / 'backtest_results.parquet')

pm = bt[bt['checkpoint'] == 0].copy()

pm['y_h'] = (pm['final_h'] > pm['final_a']).astype(int)
pm['y_d'] = (pm['final_h'] == pm['final_a']).astype(int)
pm['y_a'] = (pm['final_h'] < pm['final_a']).astype(int)

pm['model_brier'] = ((pm['model_p_home'] - pm['y_h'])**2
                   + (pm['model_p_draw'] - pm['y_d'])**2
                   + (pm['model_p_away'] - pm['y_a'])**2)

pm['market_brier'] = ((pm['market_p_home'] - pm['y_h'])**2
                    + (pm['market_p_draw'] - pm['y_d'])**2
                    + (pm['market_p_away'] - pm['y_a'])**2)

def logloss(p_h, p_d, p_a, y_h, y_d, y_a):
    eps = 1e-15
    return -(y_h * np.log(np.clip(p_h, eps, 1))
           + y_d * np.log(np.clip(p_d, eps, 1))
           + y_a * np.log(np.clip(p_a, eps, 1)))

pm['model_ll']  = logloss(pm['model_p_home'], pm['model_p_draw'], pm['model_p_away'],
                          pm['y_h'], pm['y_d'], pm['y_a'])
pm['market_ll'] = logloss(pm['market_p_home'], pm['market_p_draw'], pm['market_p_away'],
                          pm['y_h'], pm['y_d'], pm['y_a'])

print(f"n = {len(pm)}")
print(f"Model Brier:   {pm['model_brier'].mean():.4f}")
print(f"Market Brier:  {pm['market_brier'].mean():.4f}")
print(f"Gap (market-model): {pm['market_brier'].mean() - pm['model_brier'].mean():+.4f}")
print(f"  (positive = market beats model)")
print()
print(f"Model LogLoss:  {pm['model_ll'].mean():.4f}")
print(f"Market LogLoss: {pm['market_ll'].mean():.4f}")
print(f"Gap: {pm['market_ll'].mean() - pm['model_ll'].mean():+.4f}")